# synthetic_data.py

These 8 features were chosen because they represent the standard **on-chain "fingerprints"** used by real-world crypto forensic teams and AML (Anti-Money Laundering) compliance systems.

Fraudsters behave very differently from regular users in four dimensions: **volume**, **network shape**, **obfuscation**, and **timing/lifecycle**.

---

### The 8 Features & Why They Matter

| # | Feature Name | What It Tracks | Why Scammers Stand Out |
| --- | --- | --- | --- |
| 1 | `tx_count` | Total transactions sent/received | Scammers move funds rapidly through many hops to avoid detection. |
| 2 | `avg_amount` | Average transaction value | Money laundering networks often route specific batch sizes (e.g., structuring to stay under reporting limits). |
| 3 | `amount_std` | Standard deviation of amounts (variance) | Normal users spend varying amounts; scam distribution bots often move identical, uniform chunks (low variance). |
| 4 | `unique_counterparties` | How many different wallets it trades with | "Mule" or aggregator wallets touch dozens of brand-new wallets within hours to disperse stolen loot. |
| 5 | `burst_score` | Sudden spikes in transaction speed | High bursts occur when a phishing pool drains a victim or rapidly offloads hacked crypto. |
| 6 | `mixer_hop_score` | Proximity to privacy protocols (like Tornado Cash) | Normal users rarely touch mixers; high scores indicate deliberate attempts to break the audit trail. |
| 7 | `night_activity_ratio` | Transactions during off-peak/night hours | Automated draining scripts and botnets run 24/7 or coordinate across international time zones. |
| 8 | `new_wallet_ratio` | Percentage of interacting wallets created recently | Fraud rings use "burner" (freshly made) wallets to prevent historical tracing. |

---

### Why Use These in This Specific Demo?

1. **Normalized Scale ($0.0$ to $1.0$):** All features fit the same numeric range, making it straightforward for neural networks or simple classifiers to compute weights without one feature overpowering the others.
2. **Pedagogical Simplicity:** Eight features provide enough complexity to mimic real multi-dimensional fraud analysis without making the code hard to read or debug.
3. **Simulating the "Blind Spot":** When a bank has full visibility, high values across multiple features (like high `burst_score` + high `mixer_hop_score`) clearly reveal fraud. But when an isolated bank only sees 1 transaction (`visibility = 0.0`), all 8 signals collapse down to ~`0.40`, tricking the local model into seeing a harmless wallet.

A **crypto mixer** (also called a "tumbler" or "privacy protocol," like Tornado Cash) is a service designed to **break the public paper trail** on a blockchain.

---

### The Problem Mixers Solve (and Why Scammers Love Them)

Blockchains like Bitcoin or Ethereum are **completely transparent public ledgers**.

* If Wallet A sends $10,000 to Wallet B, and Wallet B sends it to an exchange, anyone in the world can open a blockchain explorer and trace the exact path.
  
* If a hacker steals $10,000, that money is "tainted." The moment it hits a regulated exchange, the exchange will see the history and freeze it.

To hide where the stolen money came from, criminals use a **mixer**.

---

### How a Mixer Works (The "Hat" Analogy)

Imagine 100 people walk into a room:

1. Everyone throws a $100 bill into a giant hat.
   
2. The hat is shaken up so all the bills are mixed together.

   
3. Everyone takes turns drawing a different $100 bill out of the hat and walking out the back door.

The total money in the room didn't change, but **it is now impossible to match who deposited which specific bill.**

In crypto:

* A scammer deposits "tainted" crypto into a smart-contract pool along with hundreds of other users.
* A short time later, they withdraw the same amount from a **brand-new, clean wallet**.
* The on-chain connection between the stolen funds and the new wallet is severed.

---

### What `mixer_hop_score` Measures in Your Code

In blockchain forensics, a **"hop"** is a single transaction step between two wallets:

* **0 hops:** Direct deposit into or withdrawal from a mixer.
* **1 hop:** Wallet A $\rightarrow$ Wallet B $\rightarrow$ Mixer.
* **2 hops:** Wallet A $\rightarrow$ Wallet B $\rightarrow$ Wallet C $\rightarrow$ Mixer.

The closer a wallet is to a known mixer address, the higher its **`mixer_hop_score`** (closer to `1.0`):

| Score | Meaning | Real-World Context |
| --- | --- | --- |
| **`0.0 - 0.2`** | Very far or zero interaction with mixers | Normal everyday user (safe). |
| **`0.8 - 1.0`** | Direct or 1-hop interaction with a mixer | Likely laundering stolen funds or evading sanctions (high risk). |

In your dataset, scam wallets are given high mixer scores because real-world fraud rings almost always route stolen crypto through mixers before attempting to cash out.

You are touching on the exact core problem of fraud detection, but there's a slight twist in **why** we squash it in this code.

We are **not** squashing the numbers to protect normal users or avoid false alarms.

We are squashing the numbers because **Bank A literally does not have the evidence.**

---

### Think of it like a CCTV Camera with a Blurry Lens

Imagine a burglar is breaking into a house:

* **The Reality (Ground Truth):** The guy is carrying crowbars, wearing a ski mask, and breaking a window. In reality, he is 100% a criminal (`base = 0.75`).
* **Bank A's Camera:** Bank A's camera is 100 feet away, pointed at the sidewalk, and covered in fog (`visibility = 0.0`).
* **What Bank A Sees:** The camera just captures a blurry figure walking on the sidewalk. To Bank A, the footage looks like a normal neighbor taking an evening walk (`0.40`).

Bank A didn't *choose* to ignore the burglar. Bank A's camera simply didn't capture the crowbar or the broken window because the criminal didn't do those things in front of Bank A.

---

### In Crypto / Banking Terms

* The scammer stole $1,000,000 using smart contract exploits, mixers, and botnets across 10 different platforms.
  
* But at **Bank A**, the scammer only did **one boring $50 transfer**.

* When Bank A calculates the features (e.g., `burst_score`, `tx_count`, `mixer_hops`), Bank A only knows about that single $50 transfer.

* Therefore, the numbers Bank A calculates for this wallet naturally look tiny and harmless (~`0.40`).

---

### Why the Formula Squashes It in Synthetic Code

Because this is a **synthetic simulation**, we don't have real blockchain ledgers running live.

We first generate what the criminal *actually* did in full (`base = 0.75`). Then, to simulate Bank A's **blind spot** (only seeing 1 tiny transaction), the math formula:

$$\text{dampened} = (\text{base} \times \text{visibility}) + 0.4 \times (1 - \text{visibility})$$

artificially drags the score down to `0.40` to recreate what Bank A's limited view would look like in the real world.

---

### How the Global Model Solves This

* Bank A's model alone says: *"I only see `0.40`, so this wallet is safe."*
* Bank B, Bank C, and Bank D saw the same ring doing mixer hops and burst transfers elsewhere.
* When the global model combines the learned patterns from all banks, it evaluates the full footprint (`visibility = 0.95`, score `~0.62+`) and realizes: **"This isn't a normal user—this is part of a coordinated ring."**

Here is the blueprint for **Function 1**: `_make_wallet_features`.

---

### What the Function Does

It generates an array of **8 numbers** (between $0.0$ and $1.0$) for a single wallet based on two inputs:

* **`label`**: `0` for safe/benign, `1` for risky/scam.
* **`visibility`**: A number from `0.0` (completely blinded) to `1.0` (fully clear).

---

### The 3 Core Operations Inside:

1. **Pick the Base Center (`loc`):**
* If `label == 0` (safe), center the distribution around **`0.35`**.
* If `label == 1` (risky), center the distribution around **`0.62`**.
* Use `rng.normal(loc=..., scale=0.22, size=8)` to generate 8 random values around that center.

2. **Clip Values (`0.0` to `1.0`):**
* Random normal values might occasionally drop below `0` or jump above `1`.
* Use `np.clip(base, 0.0, 1.0)` to keep them bounded.

3. **Apply the Dampening Formula:**
* Blend the true signal with a neutral `0.40` baseline based on visibility:

$$\text{final} = (\text{base} \times \text{visibility}) + 0.40 \times (1 - \text{visibility})$$

Yes, exactly! **`scale` is the standard deviation** ($\sigma$) of the normal (bell-curve) distribution.

---

### What `loc` vs `scale` Mean:

* **`loc` (Mean / Average):** Where the center of the bell curve sits.
* For safe wallets: `loc = 0.35`
* For scam wallets: `loc = 0.62`

* **`scale` (Standard Deviation / Spread):** How wide or spread out the values are around that center.

---

### How Changing `scale` Affects the Numbers:

* **Small `scale` (e.g., `0.05`):** Tight cluster. Almost all generated numbers will be very close to the center (e.g., between `0.30` and `0.40` for safe wallets).
* **Large `scale` (e.g., `0.22`):** Wide spread. The numbers vary realistically—some features might be `0.15`, others `0.55`—mimicking the natural variety in real user data.

---

### In Python:

```python
# Generates 8 numbers centered at 0.35 with a spread (std dev) of 0.22
base = rng.normal(loc=0.35, scale=0.22, size=8)
```

Because `scale=0.22` allows some numbers to randomly land outside the $[0, 1]$ range (like `-0.05` or `1.08`), the next line uses `np.clip(base, 0, 1)` to keep every feature strictly between `0.0` and `1.0`.

Let's trace the function line-by-line using real numbers so you can see exactly how the data flows from input to output.

---

### Imagine You Call the Function Like This:

```python
make_wallet_features(label=1, visibility=0.0, rng=rng)
```

*(Meaning: "Generate stats for a **scam wallet** (`label=1`), but make it **blinded** (`visibility=0.0`) to simulate Bank A's limited view.")*

---

### Step-by-Step Execution Trace

#### Step 1: Pick the center point

```python
loc = 0.35 if label == 0 else 0.62
```

* Since `label` is `1`, `loc` becomes **`0.62`**.
* The code knows this wallet should have dangerous/high-risk numbers.

---

#### Step 2: Draw 8 random numbers around `0.62`

```python
base = rng.normal(loc=0.62, scale=0.22, size=8)
```

The random generator creates 8 raw values with a spread of `0.22`. Let's say one of the 8 values lands on **`0.75`** (a strong high-risk signal).

---

#### Step 3: Keep numbers inside $[0.0, 1.0]$

```python
base = np.clip(base, 0, 1)
```

* If a number was `-0.05`, it becomes `0.0`.
* If a number was `1.12`, it becomes `1.0`.
* Our `0.75` stays **`0.75`**.

---

#### Step 4: The Magic Math (Apply Visibility)

```python
dampened = base * visibility + 0.4 * (1 - visibility)
```

Let's plug our actual numbers into this equation:

* `base` = **0.75**
* `visibility` = **0.0**

$$\text{dampened} = (0.75 \times 0.0) + 0.4 \times (1 - 0.0)$$

$$\text{dampened} = 0 + 0.4 \times 1 = \mathbf{0.40}$$

The strong `0.75` alarm signal got completely squashed down to **`0.40`** (which looks totally safe/normal).

---

### What if `visibility` was high (`0.90`)?

Let's test the same equation with high visibility:

$$\text{dampened} = (0.75 \times 0.90) + 0.4 \times (1 - 0.90)$$

$$\text{dampened} = 0.675 + 0.04 = \mathbf{0.715}$$

Because visibility is high, the strong alarm (`0.715`) survives, and the AI knows it's a scam!

---

#### Step 5: Return the Array

```python
return dampened
```

The function sends back the final array of 8 numbers.

It is not a universal law or an official cryptographic standard. It is a classic **linear interpolation (Lerp)** formula used everywhere in data science, game development, and graphics.

---

### 1. The Standard "Lerp" Formula

In computer science and math, whenever you want to smoothly transition between two values ($A$ and $B$) based on a slider/weight ($t$ between $0.0$ and $1.0$), you use:

$$\text{Result} = A \cdot t + B \cdot (1 - t)$$

* If $t = 1.0$ (slider 100% to $A$), you get $A$.
* If $t = 0.0$ (slider 100% to $B$), you get $B$.
* If $t = 0.5$ (slider in the middle), you get a 50/50 mix of $A$ and $B$.

---

### 2. How the Author Applied It Here

The developer simply mapped that standard interpolation formula to the demo's variables:

* **$A$ = `base`** (The real underlying risk score, e.g., $0.75$)
* **$B$ = $0.40$** (The neutral baseline value where wallets look completely safe/unremarkable)
* **$t$ = `visibility`** (How much data this specific institution can see, from $0.0$ to $1.0$)

$$\text{dampened} = (\text{base} \times \text{visibility}) + 0.40 \times (1 - \text{visibility})$$

---

### Why Pick $0.40$ Specifically?

Because safe wallets in this script center around $0.35$ with some spread, setting the "zero visibility" baseline to $0.40$ guarantees that whenever an institution is blind (`visibility = 0.0`), the numbers fall right into the harmless/average zone.

It's an intuitive simulation trick to model partial observability mathematically.

Let's break down this function piece by piece so you can see how everything fits together.

---

### What this Function Does

It generates the complete, self-contained dataset for **one specific bank** (or exchange), identified by `node_id`.

It outputs four things:

1. **`X`**: The feature matrix (all the rows of 8 numbers).
2. **`y`**: The labels array (`0` for benign, `1` for risky).
3. **`wallet_ids`**: Realistic ID strings for tracking (e.g., `"n1-benign-0"`, `"hero-3"`).
4. **`cluster_map`**: A dictionary that groups connected risky wallets together (e.g., `{"hero-0": "0x7a2...f1"}`).

---

### Step-by-Step Walkthrough

#### 1. Setup & Proportions

```python
rng = np.random.default_rng(42 + node_id)
n_risky = int(n_samples * 0.18)
n_benign = n_samples - n_risky
```

* **Node-specific Seed:** Bank 1 gets seed `43`, Bank 2 gets `44`, etc. Every bank gets unique random data.
* **82 / 18 Split:** Out of 400 samples, 18% (72) are risky, and 82% (328) are benign.

---

#### 2. Pre-allocating the Memory

```python
X = np.zeros((n_samples, N_FEATURES), dtype=np.float32)
y = np.zeros(n_samples, dtype=np.int64)
wallet_ids = []
cluster_map = {}
```

Prepares empty arrays so filling in values is fast and organized.

---

#### 3. Generating Benign (Safe) Wallets

```python
for i in range(n_benign):
    X[i] = _make_wallet_features(0, visibility=0.9, rng=rng)
    y[i] = 0
    wallet_ids.append(f"n{node_id}-benign-{i}")
```

* Generates 328 safe wallets centered around `0.35`.
* High visibility (`0.9`) means these are clear, normal transactions.
* Assigns IDs like `"n1-benign-0"`, `"n1-benign-1"`.

---

#### 4. Generating Local Risky Wallets

```python
for j in range(n_risky):
    idx = n_benign + j
    y[idx] = 1
    wallet_ids.append(f"n{node_id}-risky-{j}")
    cluster_map[wallet_ids[-1]] = f"cluster-{node_id}-{j % 5}"
    X[idx] = _make_wallet_features(1, visibility=0.9, rng=rng)
```

* Generates 72 risky wallets centered around `0.62`.
* These are clear local scams that this bank *can* easily spot on its own.
* **`j % 5` grouping:** Groups these 72 wallets into 5 local clusters (`"cluster-1-0"`, `"cluster-1-1"`, etc.).

---

#### 5. Injecting the Trap: The Hero Cluster (`include_hero=True`)

```python
if include_hero:
    hero_dampened, _, hero_y = generate_hero_cluster_views()
    hero_ids = [f"hero-{k}" for k in range(len(hero_y))]
    for wid in hero_ids:
        cluster_map[wid] = HERO_CLUSTER_ID
    X = np.vstack([X, hero_dampened])
    y = np.concatenate([y, hero_y])
    wallet_ids += hero_ids
```

This is the core of the demo:

* If `include_hero=True` (enabled for Bank 1 / Node A), we grab the **`hero_dampened`** view (14 wallets whose stats are flattened to `0.40`).
* We stack them onto the bottom of `X` and `y` using `np.vstack` and `np.concatenate`.
* We map their IDs (`"hero-0"`, `"hero-1"`, ...) to `HERO_CLUSTER_ID` (`"0x7a2...f1"`).

Bank 1 now has **414 wallets** in total. The 14 hero wallets have a true label of `1`, but their numbers look completely benign (`~0.40`).

---

### Summary of What Bank 1 Holds

| Data Chunk | Size | True Label (`y`) | Numbers (`X`) | What Bank 1 Thinks |
| --- | --- | --- | --- | --- |
| **Normal Users** | 328 | `0` (Safe) | ~`0.35` | Safe |
| **Local Scams** | 72 | `1` (Risky) | ~`0.62` | Risky (Caught locally) |
| **Hero Cluster** | 14 | `1` (Risky) | ~`0.40` | **Safe (Blind spot / Missed!)** |

Here is a visual map showing the exact shapes, numbers, and transformations of **`X`** (the feature matrix) and **`y`** (the labels) at every stage.

---

### 1. `_make_wallet_features` (A Single Wallet)

This function creates **1 row** of 8 numbers.

```text
Inputs: label = 0 (benign), visibility = 0.9

Output 1D Array (Length 8):
[ 0.36,  0.31,  0.42,  0.29,  0.38,  0.33,  0.40,  0.35 ]  <-- centered around ~0.35
```

```text
Inputs: label = 1 (scam), visibility = 0.9

Output 1D Array (Length 8):
[ 0.65,  0.71,  0.59,  0.68,  0.74,  0.63,  0.58,  0.66 ]  <-- centered around ~0.62
```

---

### 2. `generate_hero_cluster_views` (The 14 Scam Wallets)

This generates the same 14 wallets in two parallel views.

```text
                        dampened (Bank A's view)                         true_view (Global view)
                         Shape: (14, 8), float32                            Shape: (14, 8), float32
            ┌                                            ┐    ┌                                            ┐
 hero-0  -> │ 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, ...    │    │ 0.64, 0.70, 0.58, 0.67, 0.72, 0.61, ...    │
 hero-1  -> │ 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, ...    │    │ 0.59, 0.63, 0.65, 0.60, 0.69, 0.57, ...    │
  ...       │ ...   ...   ...   ...   ...   ...          │    │ ...   ...   ...   ...   ...   ...          │
 hero-13 -> │ 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, ...    │    │ 0.66, 0.58, 0.73, 0.64, 0.60, 0.68, ...    │
            └                                            ┘    └                                            ┘

                                                 hero_y
                                          Shape: (14,), int64
                                  [ 1, 1, 1, 1, 1, 1, ..., 1 ]
```

---


#### Edit: 
n_wallets: pass an explicit count to override. If None, the cluster scales with the dataset — hero_fraction of n_total_samples (min 14, so it never shrinks below the original demo's needle-in-a-haystack size).
    

### 3. `generate_institution_dataset` (Building Bank 1's Dataset)

Watch how `X` and `y` are assembled row by row, and then expanded with `np.vstack`.

#### Step A: Initialize Empty Containers

```text
X (zeros): Shape (400, 8)                y (zeros): Shape (400,)
┌                                    ┐   ┌   ┐
│ 0.0, 0.0, 0.0, 0.0, ..., 0.0       │   │ 0 │
│ 0.0, 0.0, 0.0, 0.0, ..., 0.0       │   │ 0 │
│ ...                                │   │...│
│ 0.0, 0.0, 0.0, 0.0, ..., 0.0       │   │ 0 │
└                                    ┘   └   ┘
```

#### Step B: Fill Benign (328 rows) and Risky (72 rows)

```text
Rows 0 to 327: Benign Wallets (label = 0)
Rows 328 to 399: Local Risky Wallets (label = 1)

                     X (Filled, 400 rows)                           y (400 labels)
       ┌                                             ┐             ┌   ┐
n1-b-0 │ 0.36, 0.31, 0.42, 0.29, 0.38, 0.33, ..., 0.35│ ----------> │ 0 │  (Safe)
n1-b-1 │ 0.33, 0.38, 0.30, 0.41, 0.34, 0.36, ..., 0.32│ ----------> │ 0 │  (Safe)
 ...   │ ...                                         │             │...│
n1-r-0 │ 0.65, 0.71, 0.59, 0.68, 0.74, 0.63, ..., 0.66│ ----------> │ 1 │  (Risky)
n1-r-1 │ 0.62, 0.58, 0.69, 0.64, 0.70, 0.61, ..., 0.65│ ----------> │ 1 │  (Risky)
       └                                             ┘             └   ┘
```

#### Step C: Inject the Hero Cluster (`np.vstack` & `np.concatenate`)

Because `include_hero=True`, the 14 dampened hero rows are glued to the bottom:

```text
                  FINAL X FOR BANK 1 (Shape: 414 x 8)                    FINAL y (Shape: 414)
       ┌                                                             ┐   ┌   ┐
Row 0  │ 0.36,  0.31,  0.42,  0.29,  0.38,  0.33,  0.40,  0.35       │   │ 0 │
...    │ ...    ...    ...    ...    ...    ...    ...    ...        │   │ 0 │ (328 Benign)
Row 327│ 0.34,  0.37,  0.32,  0.39,  0.35,  0.31,  0.38,  0.36       │   │ 0 │
-------┼------------------------------------------------------------┼---+---|
Row 328│ 0.65,  0.71,  0.59,  0.68,  0.74,  0.63,  0.58,  0.66       │   │ 1 │
...    │ ...    ...    ...    ...    ...    ...    ...    ...        │   │ 1 │ (72 Risky)
Row 399│ 0.61,  0.64,  0.67,  0.59,  0.70,  0.62,  0.65,  0.63       │   │ 1 │
=======╪============================================================╪===╪===╪=======================
Row 400│ 0.40,  0.40,  0.40,  0.40,  0.40,  0.40,  0.40,  0.40       │   │ 1 │ <-- THE TRAP:
...    │ ...    ...    ...    ...    ...    ...    ...    ...        │   │ 1 │ True label is 1 (scam),
Row 413│ 0.40,  0.40,  0.40,  0.40,  0.40,  0.40,  0.40,  0.40       │   │ 1 │ but features look ~0.40!
       └                                                             ┘   └   ┘
```

---

### Why this structure works for the demo

* **Bank A's AI trains on these 414 rows:** When it encounters the 14 hero wallets (`Rows 400-413`), their features (`0.40`) are almost indistinguishable from normal benign traffic (`0.35`). The local model predicts them as **safe**.
* **Global Model Evaluation:** The server feeds the **`true_view`** matrix (where those 14 wallets have features like `0.68`) into the federated model. The global model recognizes the high values and flags them as **scam**.

Here is the breakdown of the final function in the script: `generate_holdout_test_set`.

```python
def generate_holdout_test_set(n_samples: int = 300):
    """A clean, well-visible test set used only by the server to score the
    global model each round (this is fine — it's not any institution's raw
    data, just an evaluation set the server owns for the demo)."""
    rng = np.random.default_rng(999)
    n_risky = int(n_samples * 0.3)
    n_benign = n_samples - n_risky
    X = np.vstack([
        np.stack([_make_wallet_features(0, 0.95, rng) for _ in range(n_benign)]),
        np.stack([_make_wallet_features(1, 0.95, rng) for _ in range(n_risky)]),
    ]).astype(np.float32)
    y = np.concatenate([np.zeros(n_benign), np.ones(n_risky)]).astype(np.int64)
    return X, y
```

---

### What this Function Does

It generates a clean, unbiased **evaluation benchmark dataset** that lives solely on the central coordinator/server.

In machine learning, a "holdout test set" is never used for training. Its only job is to evaluate how well the **Global Federated Model** performs after each round of training.

---

### Step-by-Step Breakdown

#### 1. Setup & Class Proportions

```python
rng = np.random.default_rng(999)
n_risky = int(n_samples * 0.3)
n_benign = n_samples - n_risky
```

* **Seed `999`:** Guarantees this evaluation benchmark stays identical across every run of the demo.
* **30 / 70 Split:** Out of 300 test samples, **90 are risky (30%)** and **210 are benign (70%)**.

---

#### 2. Building `X` with High Visibility (`0.95`)

```python
X = np.vstack([
    np.stack([_make_wallet_features(0, 0.95, rng) for _ in range(n_benign)]),
    np.stack([_make_wallet_features(1, 0.95, rng) for _ in range(n_risky)]),
]).astype(np.float32)
```

* Generates 210 clean safe wallets centered around `0.35`.
* Generates 90 clean scam wallets centered around `0.62`.
* **Visibility is set to `0.95`:** This represents "ground truth" data with no artificial blind spots or blurring. We want to test whether the global model can accurately distinguish real scams from real benign users when clear data is presented.
* **`np.stack` & `np.vstack`:** Combines the lists of arrays into a single `(300, 8)` matrix.

---

#### 3. Building `y` (Labels)

```python
y = np.concatenate([np.zeros(n_benign), np.ones(n_risky)]).astype(np.int64)
```

Creates the matching ground-truth label array of length 300:

* First 210 values are `0` (Benign).
* Last 90 values are `1` (Risky).

---

### Visualizing the Holdout Test Set

```text
               Test Set Features (X)                          Test Labels (y)
               Shape: (300, 8), float32                     Shape: (300,), int64
       ┌                                             ┐             ┌   ┐
Row 0  │ 0.34, 0.38, 0.31, 0.36, 0.40, 0.32, ..., 0.35│ ----------> │ 0 │
...    │ ...                                         │             │...│ (210 Benign)
Row 209│ 0.36, 0.32, 0.39, 0.35, 0.33, 0.37, ..., 0.31│ ----------> │ 0 │
-------┼----------------------------------------------┼-------------┼---┤
Row 210│ 0.64, 0.72, 0.58, 0.66, 0.71, 0.60, ..., 0.68│ ----------> │ 1 │
...    │ ...                                         │             │...│ (90 Risky)
Row 299│ 0.61, 0.65, 0.69, 0.63, 0.67, 0.62, ..., 0.64│ ----------> │ 1 │
       └                                             ┘             └   ┘
```

---

### How All 3 Datasets Connect in the Full Demo

| Dataset | Created By | Purpose in Demo |
| --- | --- | --- |
| **Local Bank Data** (`414, 8`) | `generate_institution_dataset` | Trains Bank A's local model (contains the blinded hero cluster trap). |
| **Hero Cluster Views** (`14, 8`) | `generate_hero_cluster_views` | Evaluates Bank A's blind spot (`dampened`) vs. the Global Model's detection (`true_view`). |
| **Holdout Benchmark** (`300, 8`) | `generate_holdout_test_set` | Evaluates general classification accuracy of the global model after federated training rounds. |

Here is the visual step-by-step breakdown of how **`X`** and **`y`** are constructed inside `generate_holdout_test_set`.

---

### 1. Step A: Creating the Two Sub-Blocks of `X`

The code calls `_make_wallet_features` with high visibility (`0.95`) for benign and risky wallets separately:

```text
1. Benign Stack (210 rows, centered at ~0.35):
   ┌                                             ┐
   │ 0.34, 0.38, 0.31, 0.36, 0.40, 0.32, ..., 0.35│  <- benign sample 0
   │ 0.36, 0.32, 0.39, 0.35, 0.33, 0.37, ..., 0.31│  <- benign sample 1
   │ ...                                         │
   │ 0.35, 0.34, 0.37, 0.31, 0.38, 0.36, ..., 0.33│  <- benign sample 209
   └                                             ┘  (Shape: 210 x 8)

2. Risky Stack (90 rows, centered at ~0.62):
   ┌                                             ┐
   │ 0.64, 0.72, 0.58, 0.66, 0.71, 0.60, ..., 0.68│  <- risky sample 0
   │ 0.61, 0.65, 0.69, 0.63, 0.67, 0.62, ..., 0.64│  <- risky sample 1
   │ ...                                         │
   │ 0.59, 0.68, 0.63, 0.70, 0.66, 0.61, ..., 0.65│  <- risky sample 89
   └                                             ┘  (Shape: 90 x 8)
```

---

### 2. Step B: Stacking into the Final Matrix `X`

`np.vstack` stacks the 210 benign rows on top of the 90 risky rows into a single 2D matrix:

```text
                  FINAL MATRIX X (Shape: 300 x 8, float32)
         ┌                                                            ┐
Row 0    │ 0.34,  0.38,  0.31,  0.36,  0.40,  0.32,  0.37,  0.35    │
Row 1    │ 0.36,  0.32,  0.39,  0.35,  0.33,  0.37,  0.30,  0.31    │  210 Benign Rows
...      │ ...    ...    ...    ...    ...    ...    ...    ...     │  (Clear, low scores ~0.35)
Row 209  │ 0.35,  0.34,  0.37,  0.31,  0.38,  0.36,  0.32,  0.33    │
---------┼----------------------------------------------------------┤
Row 210  │ 0.64,  0.72,  0.58,  0.66,  0.71,  0.60,  0.65,  0.68    │
Row 211  │ 0.61,  0.65,  0.69,  0.63,  0.67,  0.62,  0.68,  0.64    │  90 Risky Rows
...      │ ...    ...    ...    ...    ...    ...    ...    ...     │  (Clear, high scores ~0.62)
Row 299  │ 0.59,  0.68,  0.63,  0.70,  0.66,  0.61,  0.64,  0.65    │
         └                                                            ┘
```

---

### 3. Step C: Creating the Label Array `y`

`np.concatenate([np.zeros(210), np.ones(90)])` creates the 1D label array matching the rows of `X`:

```text
                        FINAL ARRAY y (Shape: 300,), int64
         ┌   ┐
Index 0  │ 0 │
Index 1  │ 0 │
...      │ 0 │  <-- 210 zeros for the benign rows
Index 209│ 0 │
---------┼---┤
Index 210│ 1 │
Index 211│ 1 │
...      │ 1 │  <-- 90 ones for the risky rows
Index 299│ 1 │
         └   ┘
```

---

### Key Contrast: Holdout Set vs. Bank 1 Local Set

* **No Blinded Traps:** In `generate_institution_dataset`, the 14 hero wallets have label `1` but features squashed to `0.40`.
* **Clean Ground Truth:** In this holdout set, every `0` is clearly low (`~0.35`) and every `1` is clearly high (`~0.62`). This gives the server a fair, gold-standard test to check how accurate the Global Model is at detecting fraud.

# entity_resolution.py 

This file handles **Local Entity Resolution**—a fancy term for: *"How does a single bank group its own suspicious wallets together before scoring them?"*

---

### What is the Purpose of this Code?

In the real world, a scammer doesn't just use one wallet at a bank; they create 5, 10, or 20 burner wallets that all behave almost identically.

This code does two things locally inside the bank's own walls:

1. **Groups similar wallets into "rings" or "clusters"** based on how identical their behavioral numbers are.
2. **Calculates a single risk score** for the entire group by running those wallets through the bank's local AI model and averaging the result.

Nothing here is shared with other banks—this happens 100% on Bank A's private server.

---

### The New Library: What is `networkx` (`nx`)?

**`networkx`** is Python's standard library for working with **Graphs** (networks of nodes and edges / connections).

Think of social media:

* **Nodes (Dots):** People (e.g., Alice, Bob, Charlie).
* **Edges (Lines):** Friendships connecting two people.

In crypto and fraud forensics:

* **Nodes:** Wallets (`wallet_ids`).
* **Edges:** High behavioral similarity (if Wallet A and Wallet B behave almost identically, we draw a line connecting them).

```text
  [Wallet 1] --------- [Wallet 2]
       \                   /
        \                 /
         \               /
            [Wallet 3]
       (A 3-Wallet Cluster)
```

`networkx` makes it effortless to say: *"Find all groups of wallets that are connected to each other by these similarity lines"* using an algorithm called **Connected Components**.

Let's break down both concepts with visual diagrams and simple numbers.

---

### Part 1: The Cosine Similarity Math (`norms` & `Xn @ Xn.T`)

Instead of measuring the raw distance between points, **Cosine Similarity** measures the **angle** between two feature vectors.

```text
Identical Direction (Angle = 0°)    Perpendicular (Angle = 90°)
Cosine Similarity = 1.0            Cosine Similarity = 0.0

       Wallet B                               Wallet B
      /                                       |
     /                                        |
    /                                         |
   /_______ Wallet A                          |_______ Wallet A
```

#### Why Normalize First (`norms = np.linalg.norm(X, axis=1)`)?

The standard formula for cosine similarity between two vectors $\vec{a}$ and $\vec{b}$ is:

$$\text{Cosine Similarity} = \frac{\vec{a} \cdot \vec{b}}{\Vert{}\vec{a}\Vert{} \Vert{}\vec{b}\Vert{}}$$

If we divide every row in $X$ by its own length ($\Vert{}\vec{a}\Vert{}$), every row becomes a **unit vector** (length = 1.0).

Once vectors have length 1.0, the bottom division is no longer needed. The formula simplifies to just the dot product:

$$\text{Cosine Similarity} = \vec{a} \cdot \vec{b}$$

#### The Matrix Multiplication Trick (`sims = Xn @ Xn.T`)

Instead of using slow Python loops to compare 500 wallets one by one, multiplying the normalized matrix `Xn` by its transpose `Xn.T` computes all $500 \times 500 = 250{,}000$ pairs instantly on your CPU/GPU.

```text
       Xn (514 x 8)             Xn.T (8 x 514)             sims (514 x 514)
   ┌                    ┐     ┌                ┐     ┌                           ┐
0  │ row 0 (length 1.0) │     │ col 0   col 1  │     │ 1.00   0.91   0.45   ...  │
1  │ row 1 (length 1.0) │  @  │                │  =  │ 0.91   1.00   0.38   ...  │
   │ ...                │     │                │     │ 0.45   0.38   1.00   ...  │
   └                    ┘     └                ┘     └                           ┘
```

* Diagonal values (`sims[0, 0]`, `sims[1, 1]`) are always **1.0** (a wallet is 100% identical to itself).
* `sims[0, 1] = 0.91` means Wallet 0 and Wallet 1 have a 91% match.

---

### Part 2: How NetworkX Connects Components

Once the matrix is calculated, we draw an edge whenever similarity is $\ge 0.85$.

```python
for i in range(n):
    for j in range(i + 1, n):
        if sims[i, j] >= 0.85:
            G.add_edge(wallet_ids[i], wallet_ids[j])
```

#### What `nx.connected_components(G)` Does

Imagine four wallets evaluated by similarity:

* **Wallet 1 and Wallet 2** have similarity **0.92** $\rightarrow$ `G.add_edge('W1', 'W2')`
* **Wallet 2 and Wallet 3** have similarity **0.88** $\rightarrow$ `G.add_edge('W2', 'W3')`
* **Wallet 1 and Wallet 3** only scored **0.82** (no direct edge drawn).
* **Wallet 4** has no similarity $\ge 0.85$ with anyone (isolated).

```text
The Graph:
    [W1] ----------- [W2] ----------- [W3]           [W4] (Isolated)
             (0.92)           (0.88)
```

`nx.connected_components(G)` traces connections transitively:

* Because **W1** connects to **W2**, and **W2** connects to **W3**, they form a single chain.
* NetworkX returns the whole group as one component: `{'W1', 'W2', 'W3'}`.
* **W4** is in its own group of size 1 (`{'W4'}`).

#### Why Filter with `if len(c) > 1`?

```python
clusters = [list(c) for c in nx.connected_components(G) if len(c) > 1]
```

A cluster must have at least 2 wallets acting together. Single isolated wallets like **W4** (`len == 1`) are excluded, leaving only coordinated groups:

```python
# Final output:
[['W1', 'W2', 'W3']]
```

#### Edit:

Function 1 modification: 

Here is the exact breakdown of what changed, why your previous implementation would have crashed or frozen at 5,000 wallets, and how this new version solves the bottleneck.

---

### The Big Picture Problem: $O(N^2)$ Complexity

When you had 500 wallets, comparing all pairs required $500 \times 500 = 250{,}000$ operations.
At **5,000 wallets**, that jumps to:

$$5{,}000 \times 5{,}000 = 25{,}000{,}000 \text{ comparisons}$$

1. **Python Loop Bottleneck:** The previous version used nested Python `for` loops iterating 25 million times, which would take seconds or minutes of CPU time.
2. **RAM Bottleneck:** A single $5{,}000 \times 5{,}000$ floating-point matrix consumes substantial continuous memory, which gets worse if you scale even higher.

---

### Key Change 1: Empty Input Guard

```python
if n == 0:
    return []

```

* Prevents runtime crashes (like empty matrix divisions or index errors) if a bank has an empty dataset.

---

### Key Change 2: Fast Vectorized Branch (`n <= max_wallets`)

For datasets up to 1,500 wallets, the double `for` loop is completely eliminated in favor of C-level vectorization:

```python
sims = Xn @ Xn.T
np.fill_diagonal(sims, 0)
adj = sims >= similarity_threshold
G = nx.from_numpy_array(adj)
G = nx.relabel_nodes(G, {i: wid for i, wid in enumerate(wallet_ids)})

```

* **`np.fill_diagonal(sims, 0)`:** Sets self-similarity (a wallet compared to itself) to 0 so nodes don't create self-loops.
* **`adj = sims >= similarity_threshold`:** Instantly creates a boolean **Adjacency Matrix** (True where similarity $\ge 0.85$, False elsewhere) in a single optimized C operation.
* **`nx.from_numpy_array(adj)`:** NetworkX builds the entire graph from the boolean matrix directly without iterating node by node in Python.
* **`nx.relabel_nodes(...)`:** Converts the default integer indices (`0, 1, 2...`) back to your actual wallet ID strings.

---

### Key Change 3: Chunked Processing for Large Datasets (`n > max_wallets`)

When $N = 5{,}000$, storing the full $5000 \times 5000$ matrix is avoided by processing in slices of **500 rows** at a time:

```python
chunk = 500
for start in range(0, n, chunk):
    end = min(start + chunk, n)
    sims_chunk = Xn[start:end] @ Xn.T  # (500, 5000)
    rows, cols = np.where(sims_chunk >= similarity_threshold)
    for r, c in zip(rows, cols):
        gi, gj = start + r, c
        if gi < gj:
            G.add_edge(wallet_ids[gi], wallet_ids[gj])

```

* **Memory Bound:** Instead of a massive matrix, it only computes a slice of shape `(500, 5000)`.
* **`np.where(...)`:** Finds only the coordinate pairs that exceed $0.85$ threshold, completely ignoring the millions of pairs that don't match.
* **Global Index Mapping (`gi = start + r`):** Maps the chunk's local row index back to the global wallet index.
* **`if gi < gj`:** Ensures each edge is only added once (preventing duplicate edge work).

---

### Comparison Summary

| Metric | Old Implementation | Scaled & Chunked Version |
| --- | --- | --- |
| **Comparison Method** | Nested Python `for` loops | Vectorized matrix operations (`@` + `np.where`) |
| **Memory Footprint** | Allocates full $N \times N$ matrix | Bounded to $500 \times N$ chunks when $N > 1500$ |
| **Graph Construction** | `G.add_edge` called in Python loop | Built via `nx.from_numpy_array` or sparse index matching |
| **5,000 Wallet Scalability** | Slow / high latency | Fast, memory-safe execution |

# model.py

**Architecture & Use Case**

This code sets up a standard client-side pipeline for Federated Learning (FedAvg) on tabular data: a shared architecture across clients, weight extraction/injection for federated aggregation, weighted loss for class imbalance, and evaluation/inference helpers.

---

**1. Model: `RiskClassifier`**

* **`nn.Sequential`**: Builds a feed-forward pipeline chaining layers sequentially without needing manual calls in `forward()`.
* **Flow**: Maps $8 \rightarrow 16 \rightarrow 8 \rightarrow 2$ logits.
* **Output**: Produces 2 unnormalized logits (Class 0: safe, Class 1: risky).

---

**2. Federated Weight Synchronization**

* **`get_weights(model)`**:
  * Extracts all layer weights and biases from `model.state_dict()`, moves them to CPU, and converts them to NumPy arrays.
  * **Purpose**: Ready to be transmitted to the central server for FedAvg parameter averaging.
* **`set_weights(model, weights)`**:
  * Maps an incoming list of averaged NumPy weights back to the model's state dictionary keys and loads them with `model.load_state_dict()`.
  * **Purpose**: Updates the local client model with aggregated weights received from the central server.

---

**3. Training: `train_one_epoch`**

* **Class Weighting (`class_weights = torch.tensor([1.0, 4.5])`)**:
  * Penalizes mistakes on Class 1 (risky cases) 4.5x more than Class 0. This counters class imbalance (where ~18% are risky).
* **Training Process**:
  * Trains the whole dataset in full batches for `epochs` iterations (default 3 local epochs).
  * Executes the standard loop: `zero_grad()` $\rightarrow$ forward pass $\rightarrow$ weighted `CrossEntropyLoss` $\rightarrow$ `backward()` $\rightarrow$ `opt.step()`.

---

**4. Validation & Inference**

* **`@torch.no_grad()` Decorator**:
  * Disables PyTorch's Autograd tracking engine during evaluation to save memory and speed up computation.
* **`evaluate(...)`**:
  * Switches to `model.eval()`.
  * Computes class predictions using `out.argmax(dim=1)` and returns unweighted loss alongside classification accuracy.
* **`predict_risk_scores(...)`**:
  * Applies `torch.softmax(..., dim=1)` across the 2 logits to convert them into probabilities.
  * Slices `[:, 1]` to return just the probability of the entity being risky (Class 1) as a NumPy array.

### Edit: Model.py modification

Let's build this explanation directly on top of what you already know:

* **Your simple model:** `Linear(8, 16) -> ReLU -> Linear(16, 8) -> ReLU -> Linear(8, 2)`
* **The foundational concepts:** Tensors, `forward()`, `loss.backward()`, `opt.step()`, and `DataLoader`.

Here is the plain-English breakdown of every new piece.

---

### 1. The Big Idea: Why Change the Model at All?

Your simple model was like a narrow 3-room hallway:

```text
8 features  ──>  [ Room of 16 ]  ──>  [ Room of 8 ]  ──>  2 outputs
```

When you scaled up to 5,000 wallets, you needed a model that can find subtler patterns. If you simply make the hallway 10 rooms deep with 64 neurons each, a major issue arises: **the gradient signal gets lost**.

During backpropagation (`loss.backward()`), math gradients travel backward from the end to the front. Passing through 10 rooms of multiplication and ReLU causes the numbers to shrink to zero. The first layers never learn anything.

**ResNet (Residual Network)** solves this by adding an "express elevator" around each room so the signal can bypass the math if needed.

---

### 2. Class `ResidualBlock` Explained

```python
class ResidualBlock(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.block(x))
```

#### What is `x + self.block(x)`? (The Skip Connection)
* `x` is the input (a vector of 64 numbers).
* `self.block(x)` is what the neural network calculated (another vector of 64 numbers).
* `x + self.block(x)` simply adds the original input back to the result.

```text
               ┌───────────────────────┐
               │    Original Input x   │
               └───────────┬───────────┘
                     │           │
                     │ (skip)    ▼
                     │     [ Dense Layers & Math ]
                     │           ▼
                     │     Calculated change: block(x)
                     │           │
                     ▼           ▼
                   [ ADD TOGETHER: x + block(x) ]
                                 │
                                 ▼
                             ReLU(...)
```

**Why this works:** The network does not have to learn the entire wallet profile from scratch at every step. It only has to learn the difference (the "residual" change). If a layer has nothing useful to add, it can output 0, and $x + 0 = x$ passes the original data forward completely unharmed.

#### What is `nn.LayerNorm(dim)`?
As data flows through layers, numbers can randomly drift to $+100.0$ or $-50.0$. Extreme numbers ruin learning.

`LayerNorm` takes the 64 numbers of a single wallet row and recalibrates them so their average is 0 and their standard deviation is 1. It keeps numbers in a well-behaved range.

#### What is `nn.Dropout(0.2)`?
Neural networks can become lazy and rely heavily on one single neuron.

`Dropout(0.2)` tells PyTorch: *"During training, randomly turn off 20% of the neurons on every step."*

This forces the remaining neurons to learn useful patterns instead of relying on a single dominant connection.

---

### 3. Class `RiskClassifier` Explained

```python
class RiskClassifier(nn.Module):
    def __init__(self, n_features: int = 8, hidden_dim: int = 64, n_blocks: int = 3, dropout: float = 0.2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
        )
        self.blocks = nn.Sequential(*[ResidualBlock(hidden_dim, dropout) for _ in range(n_blocks)])
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 2),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        return self.head(x)
```

The architecture is divided into three clear functional sections:

* **The Stem (`self.stem`):**  
  Takes the raw 8 features from the wallet and expands them to 64 numbers (`hidden_dim`). This gives the network a wider canvas to discover patterns.

* **The Blocks (`self.blocks`):**  
  Stacks 3 `ResidualBlock` instances in a row.  
  * **The `*` syntax:** `[ResidualBlock(...) for _ in range(3)]` creates a Python list of 3 blocks. The `*` unrolls that list into separate arguments so `nn.Sequential(block1, block2, block3)` can chain them together.

* **The Head (`self.head`):**  
  Compresses the 64 learned numbers down:
  * `hidden_dim // 2` is integer division ($64 // 2 = 32$).
  * `Linear(64, 32) -> ReLU -> Linear(32, 2)`.
  * The final output is 2 numbers (scores for: `[Safe, Scam]`).

---

### 4. The Training Loop Changes

```python
def train_one_epoch(model: nn.Module, X: np.ndarray, y: np.ndarray, lr: float = 0.001,
                     epochs: int = 3, batch_size: int = 128):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    class_weights = torch.tensor([1.0, 4.5])
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)

    last_loss = None
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            opt.step()
            last_loss = float(loss.item())
    return last_loss


#### What is `weight_decay=1e-4` in Adam?
In optimization, "weights" are the internal numbers the model adjusts to make predictions.

If weights grow huge (e.g., $w = 5000.0$), the model overfits and reacts wildly to tiny noise.

`weight_decay=0.0001` gently penalizes large weights at every step, keeping parameter values small and stable.

#### Why use `TensorDataset` and `DataLoader`?
* **Previously (Full-batch):** You passed all 500 rows to the model at once: `out = model(xb)`.
* **Now (Mini-batching):** With 5,000 rows, updating the weights only once per epoch is too slow.
* `TensorDataset` pairs each row in `X` with its label in `y`.
* `DataLoader(..., batch_size=128, shuffle=True)` serves the data in random bites of 128 wallets at a time.
* For 5,000 wallets, the model updates its weights $5000 / 128 \approx 39$ times per epoch instead of just once.

---

### 5. The Evaluation Loop Changes

```python
@torch.no_grad()
def evaluate(model: nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 512):
    model.eval()
    xb = torch.tensor(X, dtype=torch.float32)
    yb = torch.tensor(y, dtype=torch.long)
    correct, total, loss_sum = 0, 0, 0.0
    loss_fn = nn.CrossEntropyLoss(reduction="sum")
    for i in range(0, len(xb), batch_size):
        out = model(xb[i:i + batch_size])
        yb_batch = yb[i:i + batch_size]
        preds = out.argmax(dim=1)
        correct += int((preds == yb_batch).sum().item())
        total += len(yb_batch)
        loss_sum += float(loss_fn(out, yb_batch).item())
    return loss_sum / total, correct / total
```

#### What is `reduction="sum"`?
By default, `nn.CrossEntropyLoss()` returns the average loss of the batch.

Because the 3,000 test set might not divide evenly into 512 (the last batch has fewer samples), averaging averages causes mathematical skew.

Setting `reduction="sum"` tells PyTorch to add up the raw loss of every single wallet. At the very end, `loss_sum / total` gives the exact overall average.

---

### Summary Checklist of All New Concepts

| Concept | What It Does | Why It Is Used Here |
| :--- | :--- | :--- |
| **`x + self.block(x)`** | Adds input directly to layer output (Skip path). | Keeps gradient signals from disappearing as network grows deeper. |
| **`nn.LayerNorm`** | Normalizes individual sample feature values. | Stabilizes numbers between layers without breaking federated averaging. |
| **`nn.Dropout`** | Randomly shuts off 20% of neurons during training. | Prevents model from overfitting on local bank data. |
| **`*` operator** | Unpacks a Python list into function arguments. | Chains multiple `ResidualBlock` instances inside `nn.Sequential`. |
| **`weight_decay`** | Penalizes overly large weight values in Adam. | Keeps internal parameter values small and generalizable. |
| **`DataLoader`** | Slices 5,000 rows into shuffled batches of 128. | Allows faster, more frequent learning steps per epoch. |
| **`reduction="sum"`** | Sums individual sample losses instead of averaging. | Computes exact mathematical loss across uneven evaluation chunks. |

## DP.py

Let’s strip away all jargon and build this from the absolute ground up using a real-life analogy.

---

### 1. The Story: Why We Need This File

Imagine 5 rival banks want to work together to catch fraudsters.

They agree: *"We will never show each other our customers' raw transaction data."*

Instead, each bank:
1. Takes the shared AI model.
2. Trains it in private on their own computers.
3. Looks at how the AI's internal numbers (weights) changed.
4. Sends only those changes (the update) to a central server.

---

### 2. The Danger (The "Reconstruction Leak")

Suppose Bank A has 4,999 normal customers and one extremely bizarre billionaire scammer who did a crazy $50,000,000 flash transaction.

When Bank A trains its AI, that one billionaire causes one specific internal weight in the AI to spike massively from $0.1$ to $95.4$.

When Bank A sends its update to the central server, an attacker looking at the numbers says:

> *"Wait a minute. Why did this specific weight jump by +95.3? Bank A must have an account doing this exact crazy transaction!"*

Even without seeing the raw database, the exact weight changes leak private customer activity.

---

### 3. The 2 Fixes of Differential Privacy (DP)

To stop this leak, Bank A applies two steps to its update before sending it:

#### Step A: Clipping (The "Speed Limit")
* We set a maximum rule: *"No update can be bigger than size 1.0."*
* If Bank A’s normal learning caused a tiny change of $0.3$, it passes as is.
* If that billionaire caused a massive change of $95.4$, we mathematically shrink the whole update down so its total size is capped at $1.0$.
* **Result:** No single huge customer can leave an outsized footprint on the update.

#### Step B: Gaussian Noise (The "Static Blur")
* **What is Gaussian Noise?** It's just random numbers drawn from a bell-curve (most numbers near $0.0$, a few slightly positive like $+0.03$, a few slightly negative like $-0.02$).
* Think of it like adding faint static/fuzz over a photograph.
* We sprinkle these tiny random numbers over the changes.
* **Result:** An attacker cannot tell if a $+0.04$ change came from a real customer or from the random static we added.

When all 5 banks send their slightly noisy updates, the random static cancels out on average, but individual privacy is protected.

---

### 4. Line-by-Line Code Breakdown

Now let's trace every single line of `dp.py`:

```python
def clip_and_noise_update(old_weights: list[np.ndarray], new_weights: list[np.ndarray],
                           clip_norm: float = 1.0, noise_multiplier: float = 0.05,
                           rng: np.random.Generator | None = None) -> list[np.ndarray]:
```

* **`old_weights`**: The list of NumPy arrays representing the model before local training.
* **`new_weights`**: The list of NumPy arrays representing the model after local training.
* **`clip_norm = 1.0`**: The "speed limit" (maximum size allowed).
* **`noise_multiplier = 0.05`**: The volume dial for the static noise ($5\%$ of the clip norm).

---

#### Line 1: Calculate the Raw Changes (`deltas`)

```python
deltas = [new - old for new, old in zip(new_weights, old_weights)]
```

In math and physics, Delta ($\Delta$) means "change".

If a weight was $0.50$ (old) and is now $0.58$ (new), the delta is:
$$0.58 - 0.50 = +0.08$$

We calculate this difference for every layer in the model.

---

#### Line 2: Measure the Total Length of the Changes (`total_norm`)

```python
flat = np.concatenate([d.flatten() for d in deltas])
total_norm = float(np.linalg.norm(flat)) + 1e-12
```

* **`d.flatten()`**: Flattens 2D matrices into 1D rows, and `np.concatenate` combines them into one long list of numbers.
* **`np.linalg.norm(...)`**: Calculates the geometric length (Euclidean distance) using the Pythagorean theorem:
  $$\text{total\_norm} = \sqrt{d_1^2 + d_2^2 + d_3^2 + \dots}$$
* **`+ 1e-12`**: $10^{-12}$ ($0.000000000001$). This tiny padding prevents a crash if `total_norm` happens to be zero (avoids dividing by zero in the next line).

---

#### Line 3: Calculate the Shrink Factor (`scale`)

```python
scale = min(1.0, clip_norm / total_norm)
```

Let's test with real numbers:

* **Case 1 (Small update):** Suppose $\text{total\_norm} = 0.6$.  
  $$1.0 / 0.6 = 1.66$$  
  $$\min(1.0, 1.66) = 1.0$$  
  The update is not shrunk (`scale = 1.0`).

* **Case 2 (Too large):** Suppose $\text{total\_norm} = 4.0$.  
  $$1.0 / 4.0 = 0.25$$  
  $$\min(1.0, 0.25) = 0.25$$  
  We multiply everything by $0.25$ to shrink its size back to $1.0$.

---

#### Line 4: Clip and Add Gaussian Noise

```python
noisy_deltas = []
for d in deltas:
    clipped = d * scale
    noise = rng.normal(0.0, noise_multiplier * clip_norm, size=d.shape).astype(d.dtype)
    noisy_deltas.append(clipped + noise)
```

For each layer's change array `d`:
* **`clipped = d * scale`**: Shrinks the change if it exceeded the limit.
* **`noise = rng.normal(...)`**: Creates random fuzz centered at $0.0$ with standard deviation $\sigma = 0.05 \times 1.0 = 0.05$.
* **`clipped + noise`**: Adds the static directly onto the clipped changes.

---

#### Line 5: Rebuild the Final Safe Weights

```python
return [old + nd for old, nd in zip(old_weights, noisy_deltas)]
```

Adds the sanitized changes (`noisy_deltas`) back to the starting `old_weights`.

Returns the protected model weights ready to be safely shared with the central server.

### hash_chain.py

This file implements an **Audit Log Hash Chain** (the core building block behind blockchains like Bitcoin, Git version control, and tamper-proof financial ledgers).

Let's break down the concepts, the cryptography, and every single line of code from absolute zero.

---

### 1. The Real-World Problem: The "Rogue Server"

In a federated system, multiple banks train an AI together with a central server coordinating the rounds.

Imagine this scenario:
* In Round 2, the model had an accuracy of 62% and a loss of 0.85.
* Later, a corrupt server administrator goes into the database and edits that record to say 98% accuracy to trick regulators into thinking the system was performing well.

If it is stored in a regular database (like standard SQL), nobody can prove the history was modified.

#### The Goal
We need an append-only audit log where:
1. Every training round gets sealed in a "Block".
2. If anyone changes even a single letter, number, or decimal point in any past round, the entire log mathematically breaks and sounds the alarm (`verify() == False`).

---

### 2. The Core Cryptographic Tool: SHA-256

Before understanding a hash chain, you must understand a **Cryptographic Hash Function**.

#### What is SHA-256?
Think of SHA-256 as a mathematical digital fingerprint machine:
* You feed in any piece of text (or data) of any length.
* It always spits out a fixed-length string of 64 hexadecimal characters (numbers 0-9 and letters a-f).

```text
Input Text: "Hello"
SHA-256:    185f8db32271fe25f561a6fc938b2e264306ec304eda518007d1764826381969

Input Text: "hello"  (tiny lowercase change)
SHA-256:    2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824
```

#### The 3 Magic Properties of SHA-256:
* **Deterministic:** If you feed in the exact same input tomorrow or 10 years from now, it will produce the exact same 64-character hash.
* **One-Way (Pre-image Resistance):** Given the 64-character hash, it is computationally impossible to reverse-engineer what the original input was.
* **The Avalanche Effect:** If you change a single bit (e.g., $0.85 \rightarrow 0.86$), the resulting hash changes completely and unpredictably.

---

### 3. How the "Chain" Works (The Cryptographic Glue)

How do we link blocks together so past history cannot be faked?

Every new block includes the **Hash of the Previous Block** inside its own data before computing its own hash.

```text
┌─────────────────────────┐       ┌─────────────────────────┐       ┌─────────────────────────┐
│         BLOCK 0         │       │         BLOCK 1         │       │         BLOCK 2         │
│ Prev Hash: 0000...0000  │       │ Prev Hash: 7a8f...9b12  │       │ Prev Hash: 4c2d...8e90  │
│ Round: 0                │       │ Round: 1                │       │ Round: 2                │
│ Metrics: {acc: 0.65}    │       │ Metrics: {acc: 0.78}    │       │ Metrics: {acc: 0.88}    │
│                         │       │                         │       │                         │
│ Own Hash: 7a8f...9b12   ├──────►│ Own Hash: 4c2d...8e90   ├──────►│ Own Hash: 1f3a...5b67   │
└─────────────────────────┘       └─────────────────────────┘       └─────────────────────────┘
```

#### What happens if someone tampers with Block 0?
1. The admin changes Block 0's metrics from 0.65 to 0.99.
2. Block 0's recalculation produces a completely new hash: `9999...9999` (instead of `7a8f...9b12`).
3. Now look at Block 1: Block 1 recorded that its parent's hash was `7a8f...9b12`. But Block 0's hash is now `9999...9999`.
4. The chain is broken! The mismatch immediately invalidates every subsequent block.

---

### 4. Code Breakdown: Line-by-Line

#### Imports and Setup

```python
import hashlib
import json
import time
from dataclasses import dataclass, field, asdict

GENESIS_HASH = "0" * 64
```

* **`hashlib`:** Python's built-in cryptography library that contains the `sha256` algorithm.
* **`json`:** Used to serialize data dictionaries into a standardized string format.
* **`time`:** Used to timestamp exactly when each federated round completed (`time.time()`).
* **`@dataclass`:** A clean Python decorator that automatically generates boilerplate methods like `__init__` and `__repr__` for class data structures.
* **`GENESIS_HASH = "0" * 64`:** In cryptography, the very first block in a chain is called the Genesis Block. Because it has no parent, its `prev_hash` is set to 64 zeros (`00000000000000000000...0000`).

---

#### The Block Class

```python
@dataclass
class Block:
    index: int          # Position in the chain (0, 1, 2...)
    round: int          # The federated learning round number
    timestamp: float    # Unix epoch time (e.g., 1740000000.123)
    metrics: dict       # Training results (e.g., {"accuracy": 0.88, "loss": 0.31})
    prev_hash: str      # The 64-character SHA-256 fingerprint of the prior block
    hash: str = field(default="")  # This block's own fingerprint (calculated right after creation)
```

#### How a Block Computes its Hash: `compute_hash()`

```python
def compute_hash(self) -> str:
    payload = json.dumps(
        {"index": self.index, "round": self.round, "timestamp": self.timestamp,
         "metrics": self.metrics, "prev_hash": self.prev_hash},
        sort_keys=True,
    ).encode()
    return hashlib.sha256(payload).hexdigest()

Let's dissect each sub-step:
* **`json.dumps(..., sort_keys=True)`:** Converts the dictionary into a JSON string.
  * *Why `sort_keys=True` is critical:* In Python, dictionaries might serialize keys in any order: `{"a": 1, "b": 2}` vs `{"b": 2, "a": 1}`. Even though both dictionaries contain the exact same data, their string representations are different, which would produce different hashes. Sorting the keys guarantees that identical data always yields the exact same string.
* **`.encode()`:** SHA-256 operates on raw binary bytes, not Python Unicode strings. `.encode()` converts the string to UTF-8 bytes.
* **`hashlib.sha256(payload).hexdigest()`:** Feeds the bytes into the SHA-256 engine and returns the final 64-character hexadecimal string.

---

#### The HashChain Class: Managing the Chain

```python
class HashChain:
    def __init__(self):
        self.blocks: list[Block] = []
```

Initializes an empty list to store the chain's blocks sequentially in memory.

#### Adding a New Round: `append()`

```python
def append(self, round_num: int, metrics: dict) -> Block:
    prev_hash = self.blocks[-1].hash if self.blocks else GENESIS_HASH
    block = Block(
        index=len(self.blocks),
        round=round_num,
        timestamp=time.time(),
        metrics=metrics,
        prev_hash=prev_hash,
    )
    block.hash = block.compute_hash()
    self.blocks.append(block)
    return block
```

* **`self.blocks[-1].hash if self.blocks else GENESIS_HASH`:**
  * If the chain is empty, use `GENESIS_HASH` (64 zeros) as the parent.
  * If blocks already exist, grab the hash of the latest block (`self.blocks[-1]`).
* Creates the `Block` object with the current round, timestamp, metrics, and `prev_hash`.
* Calls `block.compute_hash()` to seal the block and generate its fingerprint.
* Appends the sealed block to `self.blocks` and returns it.

---

#### Verifying Integrity: `verify()`

```python
def verify(self) -> bool:
    prev_hash = GENESIS_HASH
    for block in self.blocks:
        expected = block.compute_hash()
        if block.hash != expected or block.prev_hash != prev_hash:
            return False
        prev_hash = block.hash
    return True
```

This is the audit mechanism that regulators or dashboard clients run:
1. Starts at the beginning with `prev_hash = GENESIS_HASH`.
2. Loops through every block from index 0 to the end:
   * **Check 1 (`block.hash != expected`):** Re-runs SHA-256 on the block's current data. If someone edited a number inside `metrics`, `expected` will not match the stored `block.hash`.
   * **Check 2 (`block.prev_hash != prev_hash`):** Checks if this block points to the correct previous hash.
3. If either check fails, it immediately returns `False` (tampering detected!).
4. If all blocks pass without errors, it returns `True` (ledger is valid and untampered).

---

#### Converting to Plain Dictionaries: `as_list()`

```python
def as_list(self) -> list[dict]:
    return [asdict(b) for b in self.blocks]
```

* **`asdict(b)`:** Converts each `@dataclass Block` object into a standard Python dictionary (`{"index": 0, "round": 0, ...}`).
* Makes it simple to serialize the entire chain into JSON for frontend dashboards or exportable audit logs.

---

### Quick Summary

| Component | Responsibility |
| :--- | :--- |
| **SHA-256** | Creates an unforgeable 64-character digital fingerprint of data. |
| **`prev_hash`** | Chains each block to its predecessor so historical entries cannot be swapped or modified. |
| **`sort_keys=True`** | Ensures deterministic byte serialization for hashing. |
| **`verify()`** | Validates that no past metrics, timestamps, or linkages were altered. |

### flwr_client.py

This file is the **Federated Client** (often named `client.py`). It brings every single module you built together: `synthetic_data.py`, `entity_resolution.py`, `model.py`, and `dp.py`.

Let's break down the concepts, the new library (`flwr`), and every line of code from scratch.

---

### 1. What is Flower (`flwr`)?

**Flower (`flwr`)** is the industry-standard Python framework for Federated Learning.

#### The Client-Server Model

In Federated Learning:
* **The Server:** Coordinates the training rounds, holds the global model, and averages weights.
* **The Clients (Institutions):** The individual banks or exchanges. Each client has its own private data, trains locally, and communicates with the server.

Flower defines a standard protocol using a class called `NumPyClient`. When you inherit from `NumPyClient`, Flower expects you to implement 3 core methods:

```text
               --- FLOWER CLIENT LIFECYCLE ---

 1. get_parameters() ──> Server asks: "What are your current weights?"
 
 2. fit()            ──> Server sends global weights. 
                         Client trains on local data, applies DP noise, 
                         and sends updated weights back.
                         
 3. evaluate()       ──> Server asks: "Test the current global weights on 
                         your local test data and report loss/accuracy."
```

---

### 2. Imports and Institution Labels

```python
import numpy as np
from flwr.client import NumPyClient
from model import RiskClassifier, get_weights, set_weights, train_one_epoch, predict_risk_scores
from dp import clip_and_noise_update
from entity_resolution import cluster_wallets, local_cluster_risk_score
from synthetic_data import generate_institution_dataset, HERO_CLUSTER_ID

INSTITUTION_LABELS = {
    1: "EXCHANGE",
    2: "FORENSIC FIRM",
    3: "BANK",
    4: "BANK/EXCHANGE",
}
```

* **`NumPyClient`:** The Flower base class that automatically handles sending and receiving model weights as standard NumPy arrays over the network.
* **`INSTITUTION_LABELS`:** A human-readable dictionary that maps numerical IDs (1, 2, 3, 4) to organization types for the UI dashboard.

---

### 3. Client Initialization: `__init__`

```python
class InstitutionClient(NumPyClient):
    def __init__(self, node_id: int, n_institutions: int, dp_noise_multiplier: float = 0.05):
        self.node_id = node_id
        self.dp_noise_multiplier = dp_noise_multiplier
        self.model = RiskClassifier()

        self.X, self.y, self.wallet_ids, self.cluster_map = generate_institution_dataset(
            node_id, n_institutions, include_hero=(node_id == 1)
        )
        self.local_clusters = cluster_wallets(self.X, self.wallet_ids)
```

Let's break down what happens when Bank 1 starts up:
* **`self.model = RiskClassifier()`:** Instantiates a brand-new, randomly initialized Tabular ResNet model locally.
* **`generate_institution_dataset(...)`:** Creates this bank's private dataset (`X` features, `y` labels, `wallet_ids`, and `cluster_map`).
* **`include_hero=(node_id == 1)`:** Only Node 1 (Bank 1) gets the blinded 14-wallet hero cluster. Other banks do not get these specific rows.
* **`cluster_wallets(...)`:** Runs the NetworkX cosine-similarity grouping locally, finding all rings among its own wallets before training ever starts.

---

### 4. Method 1: `get_parameters`

```python
    def get_parameters(self, config):
        return get_weights(self.model)
```

Whenever Flower asks for the client's current weights, `get_weights(self.model)` extracts the PyTorch tensors, converts them to NumPy arrays, and returns them as a list.

---

### 5. Method 2: `fit` (The Training Round)

This is the most critical method in the whole client.

```python
    def fit(self, parameters, config):
        old_weights = [p.copy() for p in parameters]
        set_weights(self.model, parameters)

        losses = [train_one_epoch(self.model, self.X, self.y) for _ in range(3)]

        new_weights = get_weights(self.model)
        noised_weights = clip_and_noise_update(
            old_weights, new_weights, clip_norm=1.0, noise_multiplier=self.dp_noise_multiplier
        )

        metrics = {"train_loss": float(np.mean(losses)), "node_id": self.node_id}
        return noised_weights, len(self.X), metrics
```

#### Step-by-Step Flow:
1. **`old_weights = [p.copy() for p in parameters]`:**
   * The server provides the latest parameters (the global model weights).
   * We make a deep copy so we have an exact copy of what the weights looked like before training.
2. **`set_weights(self.model, parameters)`:** Loads the global weights into the local PyTorch model.
3. **`losses = [train_one_epoch(self.model, self.X, self.y) for _ in range(3)]`:**
   * Runs local training for 3 epochs on the bank's private `self.X` and `self.y`.
4. **`new_weights = get_weights(self.model)`:** Extracts the trained weights.
5. **`clip_and_noise_update(...)` (DP Step):**
   * Computes the difference ($\Delta = \text{new} - \text{old}$).
   * Clips the total update to `clip_norm=1.0`.
   * Adds calibrated Gaussian noise.

#### Return Values:
Flower requires `fit()` to return a 3-element tuple:
* **`noised_weights`:** The sanitized weights to send to the server.
* **`len(self.X)`:** Number of local training rows (used by the server to compute a weighted average).
* **`metrics`:** A dictionary of extra info (e.g., training loss and node ID) for monitoring.

---

### 6. Method 3: `evaluate` (Local Testing)

```python
    def evaluate(self, parameters, config):
        set_weights(self.model, parameters)
        from model import evaluate as eval_fn
        loss, acc = eval_fn(self.model, self.X, self.y)
        return loss, len(self.X),

* Loads the latest global model parameters from the server.
* Evaluates that global model on the bank's local data.
* Returns `(loss, num_examples, metrics_dict)`.

---

### 7. Method 4: `local_hero_cluster_score` (The Blind Spot Check)

```python
    def local_hero_cluster_score(self) -> float | None:
        hero_wallets = [w for w, c in self.cluster_map.items() if c == HERO_CLUSTER_ID]
        if not hero_wallets:
            return None
        predict_fn = lambda X: predict_risk_scores(self.model, X)
        return local_cluster_risk_score(predict_fn, self.X, self.wallet_ids, hero_wallets)
```

* Checks if this institution holds the hero cluster.
* If it is Node 1, it passes `predict_risk_scores` into `local_cluster_risk_score`.
* Because Node 1's features for the hero cluster are dampened to `0.40`, this method returns a low score (~`0.40`), proving that Node 1's isolated local model cannot detect the scam ring on its own.

---

### 8. The Factory Function: `make_client_fn`

```python
def make_client_fn(n_institutions: int, dp_noise_multiplier: float):
    def client_fn(cid: str):
        node_id = int(cid) + 1
        client = InstitutionClient(node_id, n_institutions, dp_noise_multiplier)
        return client.to_client()
    return client_fn
```

#### Why is this needed?
In Flower's simulation mode (`flwr.simulation`), you don't have to launch 4 separate physical computers or terminal windows. Flower manages all clients inside one process.

* **`cid` (Client ID):** Flower passes a string index (`"0"`, `"1"`, `"2"`, `"3"`).
* **`node_id = int(cid) + 1`:** Converts `"0"` to Node 1, `"1"` to Node 2, etc.
* **`client.to_client()`:** Converts the `NumPyClient` into a standard Flower Client object under the hood.

---

### Client Responsibilities Summary

| Action | Function Called | Data Shared With Server? |
| :--- | :--- | :--- |
| **Local Data Generation** | `generate_institution_dataset` | **NO** (Stays in local RAM) |
| **Local Clustering** | `cluster_wallets` | **NO** (Pure local computation) |
| **Model Training** | `train_one_epoch` | **NO** (Only local gradient steps) |
| **Privacy Sanitization** | `clip_and_noise_update` | **NO** (Noise added before sending) |
| **Weight Sharing** | `fit()` return value | **YES** (Only noised weight tensors) |

### client.py

Here is the detailed, step-by-step breakdown of your production-grade `client.py` script.

---

### What Makes This Different From Simulation?

In simulation mode, Python runs multiple simulated clients inside one script using virtual threads.

This new version runs as a standalone, real-world network client:
* You can launch it in a separate terminal or inside a Docker container using a CLI command like:
  ```bash
  python client.py --node-id=1 --server-address=localhost:8080
  ```
* It connects over a real TCP network socket to the central Flower server via `start_numpy_client`.
* It runs its own independent training process, ensuring total isolation between institutions.

---

### 1. CPU Thread Optimization

```python
import torch
torch.set_num_threads(1)
```

* **The Problem:** By default, PyTorch attempts to consume every available CPU core on your machine when doing linear algebra calculations. If you run 4 bank containers simultaneously on an 8-core computer, all 4 will fight over the CPU cores, causing extreme CPU throttling and context-switching overhead.
* **The Fix:** `torch.set_num_threads(1)` limits each client process to a single CPU thread. This allows 4 or more client processes to run side-by-side smoothly without resource contention.

---

### 2. Network Client Imports & Compatibility

```python
from flwr.compat.client.app import start_numpy_client
from flwr.compat.client.numpy_client import NumPyClient
```

* **`start_numpy_client`:** The networking function that initiates a real TCP socket connection, registers the client with the central Flower server at a specific IP/port (e.g., `localhost:8080`), and enters a persistent listening loop awaiting server instructions.
* **`NumPyClient`:** The base protocol handler that automatically converts network bytes into NumPy array parameter lists.

---

### 3. The Baseline Model: `solo_model` in `__init__`

```python
# Node 1 only: a persistent "no federation" baseline model
self.solo_model = RiskClassifier() if node_id == 1 else None
if node_id == 1:
    self.hero_dampened, _, _ = generate_hero_cluster_views()
```

This sets up the core comparison showcased in your demo dashboard:
* **The Problem It Solves:** How do you prove to judges or regulators that Federated Learning actually made a difference? You need a live benchmark showing what happens if Bank A remains completely isolated without federation.
* **`self.solo_model`:** A private, isolated neural network running only on Node 1 (Bank 1). It never receives global weights from the server.
* **`self.hero_dampened`:** The 14 scam wallets whose metrics are flattened to ~`0.40`. Node 1 only ever sees this blinded view locally.

---

### 4. Training Round & Solo Baseline Tracking: `fit`

```python
def fit(self, parameters, config):
    old_weights = [p.copy() for p in parameters]
    set_weights(self.model, parameters)
    loss = train_one_epoch(self.model, self.X, self.y)

    new_weights = get_weights(self.model)
    noised_weights = clip_and_noise_update(
        old_weights, new_weights, clip_norm=1.0, noise_multiplier=self.dp_noise_multiplier
    )

    metrics = {"train_loss": float(loss), "node_id": self.node_id}

    if self.node_id == 1:
        train_one_epoch(self.solo_model, self.X, self.y)
        local_hero_score = float(np.mean(predict_risk_scores(self.solo_model, self.hero_dampened)))
        metrics["local_hero_score"] = local_hero_score

    return noised_weights, len(self.X), metrics
```

#### Step-by-Step Flow:

* **Global Model Update:**
  1. Loads the global weights received from the server (`set_weights`).
  2. Trains the federated model on local data using `train_one_epoch`.
  3. Applies Differential Privacy via `clip_and_noise_update` before anything crosses the network.

* **Solo Model Training (Node 1 Only):**
  1. Trains `self.solo_model` using only Node 1's isolated data.
  2. Evaluates the solo model on `self.hero_dampened`.
  3. Since the features are squashed to $0.40$, `local_hero_score` evaluates to ~`0.38 - 0.42` (classifying them as safe).

* **Sending Metrics to Server:**
  1. Node 1 sends `local_hero_score` inside the `metrics` dictionary.
  2. The server logs this value to display the "Node A Local Risk Score" alongside the "Global Federated Risk Score" on the dashboard.

---

### 5. CLI Argument Parsing: `main()`

```python
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--node-id", type=int, required=True)
    parser.add_argument("--server-address", type=str, default="flower-server:8080")
    parser.add_argument("--dp-noise", type=float, default=0.05)
    parser.add_argument("--n-samples", type=int, default=4000)
    args = parser.parse_args()

    client = InstitutionClient(node_id=args.node_id, dp_noise_multiplier=args.dp_noise, n_samples=args.n_samples)
    print(f"[node {args.node_id}] connecting to {args.server_address} "
          f"as {INSTITUTION_LABELS.get(args.node_id, 'INSTITUTION')} "
          f"({len(client.X)} local wallets)")

    start_numpy_client(server_address=args.server_address, client=client)
```

* **`argparse`:** Standard Python module for handling command-line flags (`--node-id=1`, `--server-address=127.0.0.1:8080`).
* **`start_numpy_client`:** Opens the TCP socket connection, registers the client, and blocks execution to handle incoming round requests from the server until training completes.

---

### Summary of Client Execution

| Component | Responsibility |
| :--- | :--- |
| **`torch.set_num_threads(1)`** | Prevents CPU core contention across parallel containers. |
| **`self.solo_model`** | Maintains an isolated, non-federated baseline on Node 1. |
| **`local_hero_score`** | Reports Node 1's local score (~`0.40`) to prove the local blind spot. |
| **`start_numpy_client`** | Establishes real socket communication with the central server. |

Here are the exact, practical differences between your previous simulation client and this new network client.

---

### Core Differences at a Glance

| Feature | Previous Client (Simulation) | New Client (Real Network / Docker) |
| :--- | :--- | :--- |
| **Execution Mode** | All 4 clients ran inside one single Python process | Runs as 4 independent processes / Docker containers |
| **Communication** | Direct in-memory Python calls (`flwr.simulation`) | Real TCP network sockets (`start_numpy_client`) |
| **CPU Management** | Default PyTorch (threads fight over all CPU cores) | `torch.set_num_threads(1)` (clean parallel execution) |
| **CLI Arguments** | None (configured via hardcoded Python functions) | Configured via flags (`--node-id=1`, `--server-address`) |
| **Solo Baseline** | Calculated post-hoc via helper method | Live `self.solo_model` trained every round on Node 1 |
| **Dataset Size** | Fixed small demo size (~400–500 rows) | Scaled up via `--n-samples=4000` CLI argument |

---

### Key Upgrades Explained

#### 1. Real Network Sockets Instead of In-Memory Shortcuts
* **Previously:** Flower used `flwr.simulation.start_simulation()`. It spun up virtual clients sharing the same system memory.
* **Now:** It uses `start_numpy_client(server_address=...)`. Each bank runs on its own computer, terminal, or Docker container and talks to the server over a real IP/port (e.g., `localhost:8080`).

#### 2. CPU Thread Limiting (`torch.set_num_threads(1)`)
* When running multiple independent client processes on one machine, each process trying to use all CPU cores will choke your system.
* Locking each client to 1 thread ensures all 4 banks train simultaneously without crashing or lagging.

#### 3. Live "Solo Model" Baseline on Node 1
Node 1 now keeps two separate models in memory:
* **`self.model`:** The federated model that gets updated from the server each round.
* **`self.solo_model`:** An isolated model trained only on Node 1's local data.

Every round, it computes `local_hero_score` using `self.solo_model` and sends it in the metrics dictionary. This gives your dashboard the live comparison showing that Node A's isolated score stays low (~0.40), while the federated model learns to detect the scam.

### server.py

Here is the step-by-step breakdown of `server.py`, which acts as the central conductor for your entire federated learning network.

---

### What Does This Server Do?

In a production federated setup, the central server has two major jobs:
* **Coordinate Flower (gRPC Server on port 8080):** Listens for connecting banks (`client.py`), runs the FedAvg (Federated Averaging) strategy across rounds, averages incoming weights, tests the global model on holdout data, and seals every round in the SHA-256 hash chain.
* **Serve the Web Dashboard (FastAPI / Uvicorn on port 8000):** Runs the REST/WebSocket API in a background thread so your frontend dashboard updates in real time.

---

### 1. Imports and Architecture Setup

```python
import argparse
import threading
import numpy as np
import uvicorn
import torch
torch.set_num_threads(1)
import flwr as fl
from flwr.server.strategy import FedAvg
from flwr.common import Parameters, parameters_to_ndarrays, ndarrays_to_parameters
from model import RiskClassifier, set_weights, evaluate as eval_model, predict_risk_scores
from synthetic_data import generate_holdout_test_set, generate_hero_cluster_views
from hashchain import HashChain
from state_store import STATE
```

* **`uvicorn` & `threading`:** `uvicorn` runs the FastAPI web app. `threading.Thread` allows Python to run both the web server and the Flower coordination engine concurrently inside one process.
* **`FedAvg` (Federated Averaging):** The baseline algorithm created by Google. It collects weights $W_1, W_2, W_3, W_4$ from each bank, calculates a weighted average based on sample count, and produces $W_{\text{global}}$.
* **`parameters_to_ndarrays` & `ndarrays_to_parameters`:** Flower sends raw binary serialized buffers over the network. These utility functions convert those network bytes into standard NumPy arrays (and vice versa) so PyTorch can load them.
* **`STATE`:** A centralized, thread-safe in-memory state store where the strategy writes the latest accuracy, audit block hashes, and hero cluster comparisons so the FastAPI frontend can read them.

---

### 2. Custom Strategy: `CipherWatchStrategy`

By subclassing `FedAvg`, we intercept Flower's round lifecycle to inject our evaluation benchmark, hash-chain ledger, and dashboard reporting:

```python
class CipherWatchStrategy(FedAvg):
    def __init__(self, *args, total_rounds: int, **kwargs):
        super().__init__(*args, **kwargs)
        self.total_rounds = total_rounds
        self.global_model = RiskClassifier()
        self.X_test, self.y_test = generate_holdout_test_set()
        _, self.hero_true, _ = generate_hero_cluster_views()
        self.chain = HashChain()
        self._last_fit_metrics: dict[int, dict] = {}
```

* **`self.global_model`:** The central master model that gets updated every round.
* **`self.X_test`, `self.y_test`:** 3,000 clean, held-out evaluation samples owned by the server.
* **`self.hero_true`:** The 14 hero scam wallets with unblinded, full-visibility features (~`0.62`), used to test whether the global model can detect the scam ring.
* **`self.chain`:** An instance of `HashChain` to log and cryptographically seal every round.
* **`self._last_fit_metrics`:** A dictionary caching the latest stats reported by each individual bank.

---

### 3. Collecting Client Updates: `aggregate_fit`

```python
    def aggregate_fit(self, server_round, results, failures):
        for _, fit_res in results:
            node_id = fit_res.metrics.get("node_id")
            if node_id is not None:
                self._last_fit_metrics[int(node_id)] = dict(fit_res.metrics)

        aggregated_parameters, aggregated_metrics = super().aggregate_fit(server_round, results, failures)
        if aggregated_parameters is not None:
            set_weights(self.global_model, parameters_to_ndarrays(aggregated_parameters))
        return aggregated_parameters, aggregated_metrics
```

#### Step-by-Step Flow:
1. **`for _, fit_res in results:`**: Iterates over the replies received from all connected clients.
2. Extracts each bank's `node_id` and training metrics (including Node 1's `local_hero_score`).
3. **`super().aggregate_fit(...)`**: Calls Flower's internal FedAvg engine, which computes the weighted average of all client weights:
   $$W_{\text{global}} = \sum_{k} \frac{n_k}{N} W_k$$
4. **`set_weights(self.global_model, ...)`**: Immediately loads those freshly averaged weights into `self.global_model`.

---

### 4. Server-Side Centralized Evaluation: `evaluate`

Right after aggregating weights, Flower calls `evaluate()` to measure how good the new global model is:

```python
    def evaluate(self, server_round: int, parameters: Parameters):
        set_weights(self.global_model, parameters_to_ndarrays(parameters))
        loss, accuracy = eval_model(self.global_model, self.X_test, self.y_test)

        global_hero_score = float(np.mean(predict_risk_scores(self.global_model, self.hero_true)))
        node1_metrics = self._last_fit_metrics.get(1, {})
        local_hero_score = node1_metrics.get("local_hero_score")
```

* Evaluates `self.global_model` against the server's 3,000 held-out test samples (`loss`, `accuracy`).
* **The Core Comparison:**
  * **`global_hero_score`:** The global model evaluates the 14 hero scam wallets (`self.hero_true`). As federated rounds progress, this score climbs to ~`0.85+` (**HIGH-RISK**).
  * **`local_hero_score`:** Retrieved from Node 1's isolated model. Because Node 1 only sees dampened features, this score remains stuck at ~`0.40` (**LOW-RISK**).

---

### 5. Sealing the Audit Block & Updating Dashboard `STATE`

```python
        clusters_flagged = int(50 + server_round * 9 + accuracy * 30)

        block = self.chain.append(server_round, {
            "global_accuracy": round(accuracy, 4),
            "clusters_flagged": clusters_flagged,
        })

        STATE.update_round(
            round_num=server_round,
            total_rounds=self.total_rounds,
            accuracy=round(accuracy, 4),
            institutions=institutions_status,
            hero_cluster={
                "id": "0x7a2...f1",
                "wallet_count": 14,
                "local_score": round(local_hero_score, 2) if local_hero_score is not None else None,
                "local_label": _risk_label(local_hero_score),
                "global_score": round(global_hero_score, 2),
                "global_label": _risk_label(global_hero_score),
            },
            clusters_flagged=clusters_flagged,
            audit_block={
                "block": f"#{block.index:04d}",
                "round": block.round,
                "hash": block.hash[:12] + "…",
                "status": "VERIFIED",
            },
        )
```

* **`self.chain.append(...)`:** Mines a new block in the SHA-256 hash chain containing this round's exact metrics, linking it to the previous round's hash.
* **`STATE.update_round(...)`:** Pushes the latest metrics into the shared state object, allowing any browser viewing the dashboard to update immediately.

---

### 6. Background Thread & Execution: `main()`

```python
def run_fastapi_in_background(port: int = 8000):
    import main as api_module
    config = uvicorn.Config(api_module.app, host="0.0.0.0", port=port, log_level="warning")
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    return thread
```

* **`daemon=True`:** Runs the Uvicorn web server as a background thread that shuts down cleanly when the main program exits.

```python
def main():
    ...
    run_fastapi_in_background(port=args.api_port)

    strategy = CipherWatchStrategy(...)

    fl.server.start_server(
        server_address=args.server_address,
        config=fl.server.ServerConfig(num_rounds=args.num_rounds),
        strategy=strategy,
    )

    assert strategy.chain.verify(), "hash chain failed self-verification"
    threading.Event().wait()
```

* Starts the FastAPI web server on port 8000.
* Starts the Flower coordination server on port 8080.
* Waits for all 4 clients to connect, then executes training for `--num-rounds=20`.
* Runs `strategy.chain.verify()` to mathematically verify that the entire chain remained untampered.
* **`threading.Event().wait()`:** Keeps the process alive after training finishes so the web dashboard remains accessible for inspection.

---

### Server Architecture Overview

```text
 ┌────────────────────────────────────────────────────────────────────────┐
 │                              SERVER.PY                                 │
 │                                                                        │
 │  [Thread 1: FastAPI / Uvicorn] (:8000)                                 │
 │     └─ Serves live UI data from STATE store                            │
 │                                                                        │
 │  [Thread 2: Flower FedAvg Coordinator] (:8080)                         │
 │     ├── 1. Waits for 4 Client connections                              │
 │     ├── 2. Broadcasts global model parameters                          │
 │     ├── 3. Aggregates DP-noised client updates                         │
 │     ├── 4. Evaluates global model on 3,000 holdout samples             │
 │     ├── 5. Appends & links SHA-256 Block in HashChain                  │
 │     └── 6. Writes round metrics to STATE store                         │
 └────────────────────────────────────────────────────────────────────────┘
```

### state_store.py

Here is the complete, beginner-friendly breakdown of `state_store.py`.

---

### What Problem Does this File Solve?

In `server.py`, you have two separate threads running simultaneously:
* **Thread 1 (Flower Federated Server):** Averages neural network weights, tracks accuracy, and mints hash-chain blocks.
* **Thread 2 (FastAPI / Web Server):** Answers HTTP requests whenever a browser dashboard asks for the latest stats.

#### The Race Condition Risk
If Thread 1 is in the middle of updating data while Thread 2 tries to read it, Thread 2 might read half-updated, corrupted data (a classic multi-threading bug called a Race Condition).

`state_store.py` provides a Thread-Safe Shared Memory Store using a Mutex Lock (`threading.Lock`).

---

### 1. The Core Mechanism: `threading.Lock()`

Think of `threading.Lock()` as a single physical key to a room:
* When Thread 1 wants to update round metrics, it takes the key and locks the door (with `with self._lock:`).
* If Thread 2 (FastAPI) tries to read metrics at that exact millisecond, it must wait outside until Thread 1 finishes and unlocks the door.
* This guarantees that data is never read in a half-written or corrupted state.

---

### 2. Class Initialization: `__init__`

```python
class DemoState:
    def __init__(self):
        self._lock = threading.Lock()
        self.total_rounds = 20
        self.current_round = 0
        self.accuracy_history: list[float] = []
        self.institutions: dict[str, dict] = {}
        self.hero_cluster = {
            "id": "0x7a2...f1",
            "wallet_count": 14,
            "local_score": None,
            "local_label": None,
            "global_score": None,
            "global_label": None,
        }
        self.audit_log: list[dict] = []
        self.clusters_flagged = 0
```

* **`self._lock`:** The Mutex lock guarding all variables in this class.
* **`self.accuracy_history`:** A list that stores the accuracy of every round in sequence (e.g., `[0.65, 0.72, 0.81, ...]`), allowing the frontend to plot accuracy over time.
* **`self.institutions`:** Tracks which banks are currently connected and synced.
* **`self.hero_cluster`:** The live comparison card values for the dashboard (Node 1's isolated score vs. the Global federated score).
* **`self.audit_log`:** Holds short summaries of the verified SHA-256 blocks for the UI table.

---

### 3. Reading State: `snapshot()`

Whenever the browser hits the API endpoint `/api/state`, FastAPI calls `STATE.snapshot()`:

```python
    def snapshot(self) -> dict:
        with self._lock:
            return {
                "round": self.current_round,
                "total_rounds": self.total_rounds,
                "global_accuracy": self.accuracy_history[-1] if self.accuracy_history else None,
                "accuracy_delta": (
                    round(self.accuracy_history[-1] - self.accuracy_history[-2], 4)
                    if len(self.accuracy_history) > 1 else 0.0
                ),
                "institutions": self.institutions,
                "institutions_online": sum(1 for i in self.institutions.values() if i["status"] != "OFFLINE"),
                "institutions_total": len(self.institutions),
                "clusters_flagged": self.clusters_flagged,
                "chain_integrity": {"verified_blocks": len(self.audit_log), "total_blocks": len(self.audit_log)},
                "hero_cluster": self.hero_cluster,
            }
```

* **`with self._lock:`** Acquires the lock before reading, releases it immediately when done.
* **`self.accuracy_history[-1]`:** Python syntax for getting the last element in the list (the most recent round's accuracy).
* **`accuracy_delta`:** Calculates how much accuracy improved compared to the prior round (`accuracy_history[-1] - accuracy_history[-2]`).
* **`institutions_online`:** Counts how many institutions are actively in "SYNCED" status.

---

### 4. Writing State: `update_round()`

Called by `CipherWatchStrategy` at the end of every federated round in `server.py`:

```python
    def update_round(self, round_num: int, total_rounds: int, accuracy: float,
                     institutions: dict, hero_cluster: dict, clusters_flagged: int,
                     audit_block: dict):
        with self._lock:
            self.current_round = round_num
            self.total_rounds = total_rounds
            self.accuracy_history.append(accuracy)
            self.institutions = institutions
            self.hero_cluster = hero_cluster
            self.clusters_flagged = clusters_flagged
            self.audit_log.append(audit_block)
```

Locks the data structure, updates every field simultaneously in memory, appends the new accuracy and audit block, and releases the lock.

---

### 5. The Global Singleton: `STATE = DemoState()`

```python
STATE = DemoState()
```

By instantiating `STATE` at the module level, every other file (`server.py`, `main.py`) that imports `STATE` references the exact same memory object.

When `server.py` writes to `STATE`, `main.py` instantly sees the updated data when responding to frontend dashboard requests.

### orchestrator.py

Here is the complete, line-by-line breakdown of `demo_runner.py` (or `federated_loop.py`).

---

### What Does This File Do?

While `server.py` and `client.py` run as separate processes connecting across real network sockets (great for production and Docker), this file is your **All-In-One Interactive Demo Driver**:
* Runs a pure Python simulation of the entire federated learning loop without requiring network socket setup.
* Implements Federated Averaging (FedAvg) directly in raw NumPy mathematics.
* Steps through 20 rounds with a visual timer (`time.sleep(1.5)`), pushing live statistics to `STATE` so your web dashboard animates dynamically on demo day.

---

### 1. The Math of Federated Averaging: `federated_average()`

```python
def federated_average(weight_list: list[list[np.ndarray]], sample_counts: list[int]) -> list[np.ndarray]:
    total = sum(sample_counts)
    n_layers = len(weight_list[0])
    averaged = []
    for layer_idx in range(n_layers):
        stacked = sum(
            w[layer_idx] * (count / total)
            for w, count in zip(weight_list, sample_counts)
        )
        averaged.append(stacked)
    return averaged
```

#### How the Weighted Average Math Works:
Suppose Bank A trained on 4,000 wallets and Bank B trained on 1,000 wallets (Total = 5,000 wallets).
* Bank A's share is $\frac{4000}{5000} = 0.80$ (80%).
* Bank B's share is $\frac{1000}{5000} = 0.20$ (20%).

For each neural network layer:
$$W_{\text{global}} = (W_A \times 0.80) + (W_B \times 0.20)$$

* **`total = sum(sample_counts)`:** Computes the denominator $N_{\text{total}}$.
* **`count / total`:** Determines the weighting fraction for each bank based on its dataset size.
* **`averaged.append(stacked)`:** Accumulates the weighted matrices across all clients layer-by-layer.

---

### 2. Setting Up the Simulation: `run_demo()`

```python
def run_demo(n_institutions: int = 4, n_rounds: int = 20, dp_noise_multiplier: float = 0.05,
             round_delay_seconds: float = 1.5):
    clients = [InstitutionClient(node_id=i + 1, n_institutions=n_institutions,
                                  dp_noise_multiplier=dp_noise_multiplier)
               for i in range(n_institutions)]

    global_model = RiskClassifier()
    global_weights = get_weights(global_model)
    X_test, y_test = generate_holdout_test_set()

    solo_model = RiskClassifier()
    node_a_data = clients[0]
    hero_dampened, hero_true, _ = generate_hero_cluster_views()

    chain = HashChain()
```

* **`clients`:** Spins up 4 in-memory bank client instances (Nodes 1, 2, 3, 4).
* **`global_model`:** The master shared neural network starting with random weights.
* **`solo_model`:** An isolated baseline model on Node 1 (Bank A) that trains only on its dampened local data.
* **`chain`:** An instance of `HashChain` to cryptographically sign each completed round.

---

### 3. The 20-Round Simulation Loop

Inside `for round_num in range(1, n_rounds + 1):`:

#### Step A: Train the Isolated Solo Baseline (Node A)

```python
train_one_epoch(solo_model, node_a_data.X, node_a_data.y)
local_hero_score_before = float(np.mean(predict_risk_scores(solo_model, hero_dampened)))
```

* Trains Node A's isolated model.
* Calculates its scam risk score on `hero_dampened`. Because features are flattened to $0.40$, `local_hero_score_before` evaluates to ~`0.38 - 0.42` (**LOW-RISK**).

#### Step B: Client Training & Differential Privacy

```python
fit_results = []
for client in clients:
    weights, n_samples, metrics = client.fit(global_weights, config={})
    fit_results.append((weights, n_samples, metrics))
```

* Distributes `global_weights` to each bank.
* Each client runs local training, adds Gaussian noise via `clip_and_noise_update()`, and returns sanitized parameters.

#### Step C: Server Aggregation

```python
global_weights = federated_average(
    [w for w, _, _ in fit_results],
    [n for _, n, _ in fit_results],
)
set_weights(global_model, global_weights)
```

* Computes the weighted parameter average using `federated_average()`.
* Loads the aggregated weights back into `global_model`.

#### Step D: Global Model Evaluation

```python
_, global_accuracy = eval_model(global_model, X_test, y_test)
global_hero_score = float(np.mean(predict_risk_scores(global_model, hero_true)))
```

* Evaluates global accuracy on the 3,000 held-out test samples.
* Evaluates the 14 hero scam wallets with unblinded true features (`hero_true`). As rounds progress, `global_hero_score` rises to ~`0.85+` (**HIGH-RISK**).

---

### 4. Updating the Dashboard & Sealing the Block

```python
institutions_status = {}
for i, client in enumerate(clients):
    node_key = f"NODE_{chr(65 + i)}"
    institutions_status[node_key] = {
        "label": INSTITUTION_LABELS.get(client.node_id, "INSTITUTION"),
        "status": "SYNCED" if i != n_institutions - 1 or round_num % 3 != 0 else "SYNCING",
    }
```

* Generates status indicators for each node (`NODE_A`, `NODE_B`, `NODE_C`, `NODE_D`).
* Simulates occasional `"SYNCING"` states on Node D to make the live UI status monitors feel realistic.

```python
hero_cluster = {
    "id": HERO_CLUSTER_ID,
    "wallet_count": 14,
    "local_score": round(local_hero_score_before, 2) if local_hero_score_before is not None else None,
    "local_label": _risk_label(local_hero_score_before),
    "global_score": round(global_hero_score, 2) if global_hero_score is not None else None,
    "global_label": _risk_label(global_hero_score),
}

clusters_flagged = int(50 + round_num * 9 + (global_accuracy * 30))

block = chain.append(round_num, {
    "global_accuracy": round(global_accuracy, 4),
    "clusters_flagged": clusters_flagged,
})

STATE.update_round(
    round_num=round_num,
    total_rounds=n_rounds,
    accuracy=round(global_accuracy, 4),
    institutions=institutions_status,
    hero_cluster=hero_cluster,
    clusters_flagged=clusters_flagged,
    audit_block={
        "block": f"#{block.index:04d}",
        "round": block.round,
        "hash": block.hash[:12] + "…",
        "status": "VERIFIED",
    },
)

time.sleep(round_delay_seconds)
```

* **`chain.append(...)`:** Mines and records the SHA-256 block into the immutable ledger.
* **`STATE.update_round(...)`:** Broadcasts the updated data to the shared state store.
* **`time.sleep(round_delay_seconds)`:** Adds a 1.5-second pause between rounds so observers can watch the training metrics animate live on the dashboard.

---

### 5. Background Thread Execution: `start_background_demo()`

```python
def start_background_demo(**kwargs) -> threading.Thread:
    t = threading.Thread(target=run_demo, kwargs=kwargs, daemon=True)
    t.start()
    return t
```

Lets FastAPI start the demo loop in the background via a web button click (`POST /api/demo/start`) without freezing the HTTP server.

---

### Execution Flow Summary

```text
               ┌────────────────────────────────────────────────────────┐
               │              ROUND START (e.g., Round 7)               │
               └───────────────────────────┬────────────────────────────┘
                                           │
       ┌───────────────────────────────────┴───────────────────────────────────┐
       ▼                                                                       ▼
[Node A Isolated Model]                                             [Federated Clients 1-4]
Train solo on dampened data                                         Train locally on private data
Result: Local Hero Score = 0.40 (LOW-RISK)                          Add Differential Privacy noise
       │                                                                       │
       │                                                                       ▼
       │                                                            [federated_average()]
       │                                                            Compute weighted average weights
       │                                                                       │
       │                                                                       ▼
       │                                                            [Global Model Update]
       │                                                            Evaluate on holdout test set
       │                                                            Result: Global Hero Score = 0.88 (HIGH-RISK)
       │                                                                       │
       └───────────────────────────────────┬───────────────────────────────────┘
                                           │
                                           ▼
                                  [HashChain.append()]
                                  Mine SHA-256 Block
                                           │
                                           ▼
                                 [STATE.update_round()]
                                 Update Web Dashboard
                                           │
                                           ▼
                                   [Sleep 1.5 Seconds]
```

### main.py

Here is the step-by-step, beginner-friendly breakdown of **`main.py`**—the web API engine connecting your backend machine learning system to your frontend dashboard.

---

### What is FastAPI and Why Do We Use It?

**FastAPI** is a modern Python web framework used to build REST APIs.

* When your frontend React/Next.js/Vite dashboard needs data, it sends an HTTP request across a specific URL (an **endpoint**).
* `main.py` catches that request, pulls the latest thread-safe numbers from `STATE`, formats them as JSON, and sends them back to the browser.

---

### 1. Imports and App Initialization

```python
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

from state_store import STATE
from orchestrator import start_background_demo

app = FastAPI(title="FTIC Backend")
```

* **`FastAPI`:** Creates the central web application instance (`app`).
* **`STATE`:** The global singleton from `state_store.py` holding live metrics.
* **`start_background_demo`:** The function (from `demo_runner.py` / `orchestrator.py`) that executes the 20-round simulation in a non-blocking background thread.

---

### 2. Cross-Origin Resource Sharing (CORS)

```python
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)
```

#### What is CORS and why is this necessary?

* Web browsers have a built-in security policy: if your frontend runs on `http://localhost:5173` (Vite) and attempts to fetch data from `http://localhost:8000` (FastAPI), the browser will block the request by default because the port numbers differ.
* **`CORSMiddleware`** with `allow_origins=["*"]` tells the browser: *"It is safe to let frontend dashboards talk to this API."*

---

### 3. Demo Control Endpoint: `POST /api/demo/start`

```python
_demo_thread = None

@app.post("/api/demo/start")
def start_demo(n_institutions: int = 4, n_rounds: int = 20, round_delay_seconds: float = 2.0):
    global _demo_thread
    if _demo_thread is None or not _demo_thread.is_alive():
        _demo_thread = start_background_demo(
            n_institutions=n_institutions, n_rounds=n_rounds, round_delay_seconds=round_delay_seconds
        )
        return {"started": True}
    return {"started": False, "reason": "already running"}
```

* **The Problem It Solves:** If a user clicks the "Start Simulation" button on the UI multiple times rapidly, you don't want to spin up 10 competing training loops at once.
* **`_demo_thread.is_alive()`:** Checks if a simulation is already in progress.
* If no simulation is running: Starts the background thread and returns `{"started": True}`.
* If one is already active: Ignores the duplicate click and returns `{"started": False, "reason": "already running"}`.

---

### 4. Dashboard Metric Endpoints (JSON Feeds)

Each endpoint maps directly to a visual widget on your frontend UI:

```text
 ┌───────────────────────────┬───────────────────────────────────────────┐
 │ API Endpoint              │ Corresponding UI Component                │
 ├───────────────────────────┼───────────────────────────────────────────┤
 │ GET /api/status           │ Top KPI Metric Cards                      │
 │ GET /api/accuracy-history │ Global Accuracy Line Chart                │
 │ GET /api/institutions     │ Institution Status Grid (A, B, C, D)      │
 │ GET /api/hero-cluster     │ Isolated vs. Federated Comparison Panel   │
 │ GET /api/audit-log        │ Immutable SHA-256 Hash Chain Table        │
 └───────────────────────────┴───────────────────────────────────────────┘
```

#### A. Top KPI Cards: `GET /api/status`

```python
@app.get("/api/status")
def get_status():
    snap = STATE.snapshot()
    return {
        "global_accuracy": snap["global_accuracy"],
        "accuracy_delta": snap["accuracy_delta"],
        "institutions_online": snap["institutions_online"],
        "institutions_total": snap["institutions_total"],
        "clusters_flagged": snap["clusters_flagged"],
        "chain_integrity": snap["chain_integrity"],
        "round": snap["round"],
        "total_rounds": snap["total_rounds"],
        "live": snap["round"] > 0 and snap["round"] < snap["total_rounds"],
    }
```

* Returns current round status, live flags, total scam clusters detected, and verified block counts.

#### B. Accuracy Line Chart: `GET /api/accuracy-history`

```python
@app.get("/api/accuracy-history")
def get_accuracy_history():
    with STATE._lock:
        history = list(STATE.accuracy_history)
    return {"rounds": list(range(1, len(history) + 1)), "accuracy": history}
```

* Formats data into X/Y coordinate lists (`rounds: [1, 2, 3...]`, `accuracy: [0.65, 0.72, 0.81...]`) so libraries like Chart.js or Recharts can render the line plot immediately.

#### C. Institution Grid: `GET /api/institutions`

```python
@app.get("/api/institutions")
def get_institutions():
    return STATE.snapshot()["institutions"]
```

* Returns the status dictionary for each node (e.g., `{"NODE_A": {"label": "EXCHANGE", "status": "SYNCED"}}`).

#### D. Scam Ring Comparison: `GET /api/hero-cluster`

```python
@app.get("/api/hero-cluster")
def get_hero_cluster():
    return STATE.snapshot()["hero_cluster"]
```

* Serves the side-by-side comparison:
  * `local_score: 0.40` (`LOW-RISK`)
  * `global_score: 0.88` (`HIGH-RISK`)

#### E. Hash-Chain Audit Log Table: `GET /api/audit-log`

```python
@app.get("/api/audit-log")
def get_audit_log(limit: int = 10):
    with STATE._lock:
        log = list(STATE.audit_log)
    return log[-limit:]
```

* **`log[-limit:]`:** Slices only the most recent 10 blocks so the UI table does not become sluggish as rounds progress.

---

### 5. Running the Backend Server

To start this API server, run:

```bash
uvicorn main:app --reload --port 8000
```

* **`main:app`**: Points to the `app` instance inside `main.py`.
* **`--reload`**: Automatically restarts the server whenever you edit your Python code.
* **`--port 8000`**: Serves the API on port 8000. Interactive Swagger documentation will be available at `http://localhost:8000/docs`.